# 08 · Evaluation

Everything needed to measure the system (Plan Sections 10-12): a benchmark with an answer key,
a runner for every ablation, an LLM judge, statistics, and two evaluations of their own for the
scraping pipeline and the voice.

| Part | Module | What it measures |
|---|---|---|
| Benchmark | `src/benchmark.py` | 100 synthetic listings (40 clean, 30 traps, 30 decoys; dev/test), fixed prices, 50 queries in MACDF's mix and 5 memory scripts, with an answer key from an oracle that only uses the ground truth |
| Runner | `src/evaluation.py` | each variant (below) on the queries: top-1/top-3 accuracy, correct refusals, policy violations, traps recommended, scam recall, false positives on honest decoys, arithmetic errors, latency, tokens, cost |
| Judge | `src/evaluation.py` | naturalness and helpfulness of each reply (gpt-oss-120b, another family than Gemini), checked against two people's ratings of 20 replies |
| Statistics | `src/evaluation.py` | 95% bootstrap intervals over queries, each variant against the full system (paired change in accuracy, exact McNemar test), results by query type |
| Pipeline | `src/eval_pipeline.py` | the scrape's health, the scope decisions (precision, recall), each identity field's accuracy, price coverage, time and cost per listing, against hand labels |
| Voice | `src/eval_voice.py` | word error rate and entity recall of Whisper on spoken requests in 4 voices, with and without the vocabulary prompt; spoken vs typed top picks; H5 latency |

Variants (`evaluation.variants` in config.yaml), each one change against `full`:

| Variant | What changes | Plan |
|---|---|---|
| `full` | nothing: the four agents as built | |
| `single_agent` | one LLM call that sees every listing with the same tool outputs | single-agent baseline |
| `frontier_single` | that single pass on the frontier model (paid) | ablation 6 |
| `small_model` | the four agents on gpt-oss-20b | ablation 5 |
| `no_reflection` | no price or scam checks before ranking | ablation 3 |
| `no_rag` | flag explanations without retrieval | |
| `no_clarify` | never asks a clarifying question | |
| `no_number_check` | the reply isn't checked | |
| `template_reply` | fixed wording, no LLM for the reply | |
| `no_memory` | nothing carried between turns | |
| `plan_scoring` | Plan v2's 3-part utility | |

Commands (from the repo root):

```bash
uv run python -m src.evaluation run --variants all --split dev --model offline   # wiring check, free
uv run python -m src.evaluation run --variants full,single_agent --split dev --model small --label pilot
uv run python -m src.evaluation estimate eval/results/<pilot folder> --repeats 3
uv run python -m src.evaluation run --variants all --split test --repeats 3 --max-cost 25 --label final
uv run python -m src.evaluation judge eval/results/<folder>      # then: stats, rating-sheet, agreement
uv run python -m src.eval_pipeline score data/samples/sample_raw_listings.json eval/data/sample_raw_labels.csv
uv run python -m src.eval_voice stt --split dev --limit 10       # Edge-TTS + Groq Whisper, free tiers
```

Repeats are part of the LLM cache key: repeats 0-2 are real calls, a re-run of the same repeat is
free. A cached call takes no time, so rows with cached calls are left out of the latency numbers.

In [ ]:
#| default_exp evaluation

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib.util
import os
import sys
from pathlib import Path

# Stop early with a clear message if VS Code picked the wrong kernel.
missing = [
    m for m in ("yaml", "dotenv", "langgraph", "pydantic") if not importlib.util.find_spec(m)
]
if missing:
    raise ModuleNotFoundError(
        f"This kernel ({sys.executable}) is not the project environment; missing {missing}. "
        "Run `uv sync` in the repo, then click the kernel name at the top right of the "
        "notebook -> Select Another Kernel -> Python Environments -> .venv (Python 3.14)."
    )

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets this notebook import the src package

LIVE = False  # True: real Gemini calls with the keys in .env. False: offline, free, no keys.
os.environ["AGENTIC_LLM"] = "primary" if LIVE else "offline"

# The benchmark (`src/benchmark.py`)

In [ ]:
#| export benchmark
"""The evaluation benchmark (Section 10): a fixed pool of 100 synthetic listings, their market
prices, 50 buyer queries with an answer key, and 5 multi-turn memory scripts.

Why synthetic: an answer key needs listings whose truth is known (which card, which condition,
scam or not). The real snapshot gets the same treatment once it is hand-checked; the runner
takes any pool and query file in this format.

- 10 real printings (Base Set, 151, Evolving Skies, Silver Tempest, Surging Sparks), each with 4
  clean listings, 3 traps (scams) and 3 decoys (honest but odd: cheap because damaged, urgent
  but real, "not a proxy", a stock photo plus real ones, "custom sleeve included").
- Dev/test split: dev traps use the red-flag knowledge base's own wording; test traps use new
  wording ("premium print", "Apple Cash up front", "reach me on Signal"), so the test split
  measures whether scam detection generalizes rather than whether it memorized its keywords.
- Prices are fixed for the benchmark, close to TCGplayer in November 2026, so results don't
  drift when the market moves.
- Queries follow MACDF's mix: 15 simple, 25 complex (buyer type, condition, clarification,
  traps as bait, nothing fits), 10 consultative (price checks).
- The answer key comes from an oracle that uses only the ground truth (the card, its true
  condition, scam or not, landed cost) and its own persona weights, never the system's risk
  score or listing score, so the test isn't circular. Acceptable top picks are the valid
  listings within `ORACLE_BAND` of the best oracle score.
"""

import json
import random
from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path
from typing import Any

from src.config import repo_path
from src.pipeline import MarketData
from src.schemas import CONDITION_ORDER, CardListing
from src.tools.landed_cost import compute_landed_cost
from src.tools.risk import parse_red_flags

## The listings

10 real printings, each with 4 clean listings (one a genuinely good deal: about 85% of market, meetup, photos), 3 traps and 3 decoys. Dev traps use the red-flag knowledge base's wording; test traps use new wording. Each spec says why a trap or decoy is one.

In [ ]:
#| export benchmark
REFERENCE_DATE = date(2026, 11, 25)  # listing dates in the files are relative to this day

PRINTINGS: dict[str, dict[str, Any]] = {
    "charizard_base": {"card_name": "Charizard", "set_name": "Base Set", "set_code": "base1",
                       "number": "4", "variant": "holo", "price_key": "holofoil", "market": 420.0},
    "blastoise_base": {"card_name": "Blastoise", "set_name": "Base Set", "set_code": "base1",
                       "number": "2", "variant": "holo", "price_key": "holofoil", "market": 160.0},
    "venusaur_base": {"card_name": "Venusaur", "set_name": "Base Set", "set_code": "base1",
                      "number": "15", "variant": "holo", "price_key": "holofoil", "market": 120.0},
    "mewtwo_base": {"card_name": "Mewtwo", "set_name": "Base Set", "set_code": "base1",
                    "number": "10", "variant": "holo", "price_key": "holofoil", "market": 50.0},
    "pikachu_base": {"card_name": "Pikachu", "set_name": "Base Set", "set_code": "base1",
                     "number": "58", "variant": "normal", "price_key": "normal", "market": 12.0},
    "charizard_ex_151": {"card_name": "Charizard ex", "set_name": "151", "set_code": "sv03.5",
                         "number": "199", "variant": "holo", "price_key": "holofoil",
                         "market": 115.0},
    "mew_ex_151": {"card_name": "Mew ex", "set_name": "151", "set_code": "sv03.5",
                   "number": "205", "variant": "holo", "price_key": "holofoil", "market": 60.0},
    "umbreon_es": {"card_name": "Umbreon VMAX", "set_name": "Evolving Skies", "set_code": "swsh7",
                   "number": "215", "variant": "holo", "price_key": "holofoil", "market": 1250.0},
    "lugia_st": {"card_name": "Lugia V", "set_name": "Silver Tempest", "set_code": "swsh12",
                 "number": "186", "variant": "holo", "price_key": "holofoil", "market": 110.0},
    "pikachu_ex_ss": {"card_name": "Pikachu ex", "set_name": "Surging Sparks", "set_code": "sv08",
                      "number": "238", "variant": "holo", "price_key": "holofoil",
                      "market": 320.0},
}  # fmt: skip

CITIES = {"Royal Oak": 12, "Dearborn": 10, "Warren": 14, "Southfield": 15, "Troy": 18,
          "Livonia": 20, "Sterling Heights": 22, "Canton": 25, "Novi": 28, "Pontiac": 30,
          "Ypsilanti": 33, "Ann Arbor": 40, "Toledo, OH": 58, "Flint": 65, "Lansing": 85,
          "Grand Rapids": 155}  # fmt: skip

# (printing, kind+n, split, condition, price, p(ickup)/s(hip), shipping, city, photos, days
#  listed, public meetup, title, description, why it is a trap or a decoy)
LISTING_SPECS: list[tuple] = [
    # ------------------------------------------------------------------ Charizard, Base Set
    ("charizard_base", "c1", "dev", "NM", 370, "p", None, "Royal Oak", 4, 5, True,
     "Charizard 4/102 Base Set holo - NM",
     ("Unlimited holo, sharp corners, clean holo, no whitening. Can meet at the Royal Oak police "
     "station lobby."), None),
    ("charizard_base", "c2", "test", "NM", 430, "s", 5.0, "Grand Rapids", 6, 12, False,
     "Base Set Charizard 4/102 NM holo",
     ("Near mint, centered, sleeved and toploaded since I opened it. Ships tracked in a bubble "
     "mailer, PayPal goods and services."), None),
    ("charizard_base", "c3", "dev", "MP", 270, "p", None, "Warren", 3, 20, True,
     "Charizard holo base set MP",
     ("Moderate play: edge wear and a light scratch on the holo, no bends. Meet at the Target in "
     "Warren."), None),
    ("charizard_base", "c4", "test", "LP", 365, "p", None, "Ann Arbor", 2, 45, False,
     "Charizard 4/102 unlimited LP",
     "Pulled it as a kid and kept it in a binder. Light corner wear. Pickup in Ann Arbor.", None),
    ("charizard_base", "t1", "dev", "NM", 190, "s", None, "Lansing", 1, 2, False,
     "Charizard Base Set 4/102 holo MINT",
     "Perfect condition. Zelle only, pay first and I ship the same day. No meetups.",
     "payment first through Zelle, ships only, 45% of market"),
    ("charizard_base", "t2", "dev", "NM", 410, "s", None, "Flint", 2, 6, False,
     "Charizard holo 4/102 near mint",
     "Comes with a case. Friends and family only to keep the price down.",
     "PayPal friends and family: no buyer protection"),
    ("charizard_base", "t3", "test", "NM", 300, "p", None, "Pontiac", 3, 3, False,
     "Base Set Charizard NM holo",
     "Gorgeous card, looks brand new. It's a premium print so the colors really pop. Price firm.",
     "a 'premium print' is a reproduction, not a real card"),
    ("charizard_base", "d1", "dev", "HP", 150, "p", None, "Dearborn", 5, 3, True,
     "Charizard base set HP - crease",
     ("Honest listing: one crease across the top corner and heavy edge wear, see photos. Priced "
     "for the damage. Not a proxy, I pulled it from my own booster in 1999."),
     "cheap because damaged, said plainly; 'not a proxy'"),
    ("charizard_base", "d2", "dev", "LP", 230, "p", None, "Livonia", 4, 1, True,
     "Charizard 4/102 LP - need it gone today",
     ("Moving out of state this weekend, need it gone today, so it's priced well under market. "
     "Happy to meet at the Livonia mall."), "urgent and cheap, but a real moving sale"),
    ("charizard_base", "d3", "test", "NM", 395, "p", None, "Southfield", 1, 30, False,
     "Charizard holo NM, cash only",
     ("Cash only, no apps. One photo because my phone camera is cracked, but you can look at it "
     "in person before paying."), "one photo and cash only, but inspection in person"),
    # ------------------------------------------------------------------ Blastoise, Base Set
    ("blastoise_base", "c1", "dev", "NM", 138, "p", None, "Sterling Heights", 5, 8, True,
     "Blastoise 2/102 Base Set holo NM", ("Near mint, clean surface. Meet at the Sterling Heights "
     "library."), None),
    ("blastoise_base", "c2", "test", "LP", 130, "p", None, "Troy", 3, 6, True,
     "Blastoise base set holo LP", "Light whitening on the back. Meet at the Troy library.", None),
    ("blastoise_base", "c3", "dev", "MP", 105, "p", None, "Sterling Heights", 2, 15, False,
     "Blastoise holo MP", "Moderate wear, small scuff on the holo. Pickup only.", None),
    ("blastoise_base", "c4", "test", "LP", 140, "s", 5.0, "Toledo, OH", 4, 25, False,
     "Base Set Blastoise 2/102 LP", "Lightly played, binder kept. Ships with tracking.", None),
    ("blastoise_base", "t1", "dev", "NM", 70, "s", None, "Flint", 1, 1, False,
     "Blastoise 2/102 holo NM", ("Must sell fast. Cash App only, pay first, then I ship. Serious "
     "buyers only."), "Cash App up front, 44% of market"),
    ("blastoise_base", "t2", "dev", "LP", 135, "p", None, "Pontiac", 2, 4, False,
     "Blastoise holo base LP", "Text me at 313-555-0142 for more photos, I don't check this app.",
     "moves the deal off the platform"),
    ("blastoise_base", "t3", "test", "NM", 120, "s", None, "Lansing", 2, 2, False,
     "Blastoise base set holo, NM", ("My brother is stationed abroad so he'll mail it from there "
     "once the payment clears."), "seller abroad, ships after payment"),
    ("blastoise_base", "d1", "dev", "MP", 75, "p", None, "Ypsilanti", 4, 7, True,
     "Blastoise base set holo MP - first come first served",
     ("Lots of interest so first come first served, but I'll hold it for whoever confirms a time. "
     "Meet at the Ypsi library."), "urgency wording, but a normal local sale"),
    ("blastoise_base", "d2", "test", "NM", 150, "s", 5.0, "Grand Rapids", 2, 9, False,
     "Blastoise 2/102 NM - PSA 10 worthy",
     ("Raw card, not graded, but sharp corners and a clean surface. PSA 10 worthy in my opinion. "
     "Ships in a toploader with tracking."), "mentions PSA but is an honest raw card"),
    ("blastoise_base", "d3", "test", "LP", 95, "p", None, "Pontiac", 3, 10, True,
     "Blastoise holo LP, custom sleeve included",
     ("Comes in a custom sleeve and toploader. Light wear on the back. Meet at the Pontiac police "
     "station."), "'custom' sleeve, a real card"),
    # ------------------------------------------------------------------ Venusaur, Base Set
    ("venusaur_base", "c1", "dev", "NM", 104, "p", None, "Novi", 4, 10, True,
     "Venusaur 15/102 Base Set holo NM", ("Near mint, sleeved since the 90s. Meet at the Novi "
     "library."), None),
    ("venusaur_base", "c2", "test", "NM", 125, "s", 4.0, "Flint", 4, 4, False,
     "Venusaur base set holo NM", "Near mint, sleeved. Ships tracked.", None),
    ("venusaur_base", "c3", "dev", "MP", 80, "p", None, "Canton", 2, 50, False,
     "Venusaur holo MP", "Played, some scratches on the holo. Pickup in Canton.", None),
    ("venusaur_base", "c4", "test", "LP", 100, "p", None, "Dearborn", 5, 2, True,
     "Venusaur base set 15/102 LP", ("Light wear, no creases. Meet at the Dearborn police "
     "station."), None),
    ("venusaur_base", "t1", "dev", "NM", 45, "s", None, "Lansing", 1, 1, False,
     "Venusaur holo base set NM", ("Brand new condition. Deposit to hold it, the rest when it "
     "ships."), "deposit before shipping, 38% of market"),
    ("venusaur_base", "t2", "dev", "LP", 98, "s", None, "Toledo, OH", 2, 5, False,
     "Venusaur 15/102 LP", "Shipping only, will ship after payment. No local.",
     "ships after payment only"),
    ("venusaur_base", "t3", "test", "NM", 90, "p", None, "Warren", 2, 3, False,
     "Venusaur 15/102 holo NM", ("Fan-favourite reproduction with a holo finish, "
     "indistinguishable from the original."), "a reproduction"),
    ("venusaur_base", "d1", "dev", "NM", 85, "p", None, "Royal Oak", 6, 1, True,
     "Venusaur base set NM - my kid's collection",
     ("Selling my son's old cards. Near mint, stored in a binder. I'm not sure of the value so I "
     "priced it to sell; meet at the Royal Oak library."), ("cheap because the seller doesn't "
     "know the value")),
    ("venusaur_base", "d2", "dev", "LP", 70, "p", None, "Warren", 3, 5, True,
     "Venusaur LP - price negotiable", ("Light wear. Price negotiable, I'm on here a lot and reply "
     "fast."), "low and negotiable, a normal sale"),
    ("venusaur_base", "d3", "test", "HP", 40, "s", 4.0, "Lansing", 4, 8, False,
     "Venusaur base set HP, bent", ("Visible bend in the middle, honest HP. As is, no returns, "
     "but you can see every photo."), "damaged and sold as is, said plainly"),
    # ------------------------------------------------------------------ Mewtwo, Base Set
    ("mewtwo_base", "c1", "dev", "NM", 43, "p", None, "Southfield", 4, 7, True,
     "Mewtwo 10/102 Base Set holo NM", ("Near mint, from a binder. Meet at the Southfield "
     "library."), None),
    ("mewtwo_base", "c2", "test", "LP", 40, "s", 4.0, "Lansing", 3, 12, False,
     "Mewtwo base set holo LP", "Light play. Ships with tracking.", None),
    ("mewtwo_base", "c3", "dev", "LP", 45, "p", None, "Troy", 2, 30, False,
     "Mewtwo holo LP", "Lightly played. Pickup in Troy.", None),
    ("mewtwo_base", "c4", "test", "MP", 30, "p", None, "Livonia", 3, 3, True,
     "Mewtwo base set MP", "Moderate edge wear. Meet at the Livonia mall.", None),
    ("mewtwo_base", "t1", "dev", "NM", 18, "s", None, "Flint", 1, 1, False,
     "Mewtwo 10/102 holo NM", "Payment first via Venmo only, I ship the next day.",
     "Venmo up front, 36% of market"),
    ("mewtwo_base", "t2", "dev", "NM", 48, "p", None, "Pontiac", 1, 4, False,
     "Mewtwo base set holo", ("Selling on behalf of my nephew's school fundraiser, he picked it out "
     "of a stack. Not sure which set."), "acting for someone else, card identity vague"),
    ("mewtwo_base", "t3", "test", "NM", 35, "s", 4.0, "Toledo, OH", 2, 2, False,
     "Mewtwo base set NM holo", "Pay me through a gift voucher and I'll post it the same day.",
     "gift-voucher payment"),
    ("mewtwo_base", "d1", "dev", "NM", 34, "p", None, "Ann Arbor", 5, 2, True,
     "Mewtwo base set NM - cheap, need gone",
     ("Clearing out duplicates before I move, need gone today. Near mint, meet at the Ann Arbor "
     "police station."), "urgent and cheap, but a real move"),
    ("mewtwo_base", "d2", "dev", "LP", 30, "s", 4.0, "Toledo, OH", 4, 6, False,
     "Mewtwo holo LP", ("The first picture is a stock image for the set, the rest are the actual "
     "card. Light wear on the corners."), "a stock photo first, then real photos"),
    ("mewtwo_base", "d3", "test", "NM", 47, "p", None, "Dearborn", 2, 14, False,
     "Mewtwo 10/102 NM, overseas buyers welcome",
     "Local pickup in Dearborn, and I also ship to overseas buyers at cost.",
     "mentions overseas, but a local seller"),
    # ------------------------------------------------------------------ Pikachu, Base Set
    ("pikachu_base", "c1", "dev", "NM", 10, "p", None, "Southfield", 3, 9, True,
     "Pikachu 58/102 Base Set NM", "Near mint yellow cheeks. Meet at the Southfield library.",
     None),
    ("pikachu_base", "c2", "test", "LP", 10, "p", None, "Royal Oak", 3, 4, True,
     "Pikachu base set LP", "Light play. Meet at the Royal Oak library.", None),
    ("pikachu_base", "c3", "dev", "LP", 9, "p", None, "Canton", 2, 35, False,
     "Pikachu 58/102 LP", "Lightly played. Pickup in Canton.", None),
    ("pikachu_base", "c4", "test", "NM", 12, "p", None, "Sterling Heights", 4, 1, True,
     "Pikachu base set 58/102 NM", "Near mint, sleeved. Meet at the Sterling Heights library.",
     None),
    ("pikachu_base", "t1", "dev", "NM", 4, "s", None, "Lansing", 1, 1, False,
     "Pikachu base set 58/102 NM red cheeks", "Rare red cheeks error! Wire transfer only.",
     "wire transfer, impossible price"),
    ("pikachu_base", "t2", "dev", "NM", 12, "s", None, "Flint", 1, 3, False,
     "Pikachu 58/102 NM", "Shipping only, no meetups. Send payment first and I'll post it.",
     "payment first, ships only"),
    ("pikachu_base", "t3", "test", "NM", 8, "p", None, "Warren", 2, 2, False,
     "Pikachu base set NM", ("I printed it myself on premium card stock, looks just like the "
     "original."), "a home-printed copy"),
    ("pikachu_base", "d1", "dev", "NM", 7, "p", None, "Warren", 5, 3, True,
     "Pikachu base set NM - first come first serve",
     "Have a few extra Pikachus, first come first serve at $7 each. Meet at the Warren library.",
     "cheap duplicates, a normal sale"),
    ("pikachu_base", "d2", "test", "LP", 6, "s", 1.0, "Grand Rapids", 3, 7, False,
     "Pikachu LP yellow cheeks", ("Light play. Ships in a penny sleeve and toploader in a plain "
     "white envelope."), "cheap common, shipped plainly"),
    ("pikachu_base", "d3", "test", "HP", 3, "p", None, "Ypsilanti", 3, 12, True,
     "Pikachu base set HP - well loved", ("Well loved, creases and a small tear. Free with any "
     "other purchase, $3 alone."), "very cheap because damaged"),
    # ------------------------------------------------------------------ Charizard ex, 151
    ("charizard_ex_151", "c1", "dev", "NM", 99, "p", None, "Troy", 4, 6, True,
     "Charizard ex 199/165 151 SIR NM", ("Pulled from a booster bundle, straight into a sleeve. "
     "Meet at the Troy library."), None),
    ("charizard_ex_151", "c2", "test", "NM", 112, "s", 5.0, "Grand Rapids", 5, 3, False,
     "Charizard ex SIR 151 NM", "Near mint, ships tracked in a toploader.", None),
    ("charizard_ex_151", "c3", "dev", "LP", 95, "p", None, "Novi", 3, 14, False,
     "Charizard ex 151 special illustration LP", ("Light whitening on one corner. Pickup in "
     "Novi."), None),
    ("charizard_ex_151", "c4", "test", "NM", 120, "p", None, "Dearborn", 6, 2, True,
     "Charizard ex 199/165 NM", "Perfect centering. Meet at the Dearborn police station.", None),
    ("charizard_ex_151", "t1", "test", "NM", 70, "s", None, "Lansing", 2, 1, False,
     "Charizard ex 199/165 SIR 151 NM", ("Fresh pull, sleeved. I can only take Apple Cash up "
     "front, then it ships."), "payment up front by an unprotected app"),
    ("charizard_ex_151", "t2", "test", "NM", 105, "s", None, "Flint", 2, 4, False,
     "Charizard ex SIR 151", ("Pick up is not possible, I'm out of town for work; I'll send it "
     "once you've paid."), "seller away, ships after payment"),
    ("charizard_ex_151", "t3", "dev", "NM", 60, "s", None, "Toledo, OH", 1, 2, False,
     "Charizard ex 199/165 NM", "Replica of the special illustration, high quality, ships fast.",
     "a replica"),
    ("charizard_ex_151", "d1", "dev", "LP", 70, "p", None, "Southfield", 4, 2, True,
     "Charizard ex 151 SIR LP - need it gone today",
     ("Small whitening on two corners. Need it gone today for rent, meet at the Southfield "
     "police station."), "urgent, but a real local sale"),
    ("charizard_ex_151", "d2", "test", "NM", 90, "p", None, "Livonia", 2, 10, False,
     "Charizard ex 199/165 NM, not a proxy",
     ("People keep asking: it's not a proxy, it's from a booster box I opened. Can meet at a card "
     "shop so you can check it."), "'not a proxy'"),
    ("charizard_ex_151", "d3", "test", "NM", 100, "s", 5.0, "Toledo, OH", 3, 5, False,
     "Charizard ex 151 SIR - shipping only",
     "I don't do meetups, but it ships the same day with tracking and PayPal goods and services.",
     "shipping only, but with buyer protection"),
    # ------------------------------------------------------------------ Mew ex, 151
    ("mew_ex_151", "c1", "dev", "NM", 52, "p", None, "Dearborn", 4, 5, True,
     "Mew ex 205/165 151 SIR NM", "Near mint, sleeved. Meet at the Dearborn library.", None),
    ("mew_ex_151", "c2", "test", "LP", 50, "p", None, "Warren", 3, 8, True,
     "Mew ex 151 gold LP", "Light wear on the back. Meet at the Warren library.", None),
    ("mew_ex_151", "c3", "dev", "NM", 58, "p", None, "Royal Oak", 5, 20, True,
     "Mew ex SIR 205/165 NM", "Sleeved since pulled. Meet at the Royal Oak library.", None),
    ("mew_ex_151", "c4", "test", "MP", 40, "p", None, "Pontiac", 2, 40, False,
     "Mew ex 151 MP", "Some scratches on the surface. Pickup only.", None),
    ("mew_ex_151", "t1", "test", "NM", 35, "p", None, "Pontiac", 2, 2, False,
     "Mew ex 205/165 gold NM", "Please send a $20 hold fee so I don't sell it to someone else.",
     "asks for a hold fee before meeting"),
    ("mew_ex_151", "t2", "test", "NM", 55, "s", None, "Flint", 2, 3, False,
     "Mew ex SIR 151 NM", "Reach me on Signal, I'm rarely on Facebook.",
     "moves the deal to another app"),
    ("mew_ex_151", "t3", "dev", "NM", 30, "s", None, "Lansing", 1, 1, False,
     "Mew ex 205/165 NM", "Cashapp only, pay first, ships in 2 days.", "Cash App up front"),
    ("mew_ex_151", "d1", "dev", "NM", 40, "p", None, "Canton", 4, 2, True,
     "Mew ex 151 SIR NM - quick sale", ("Need gone today, first come first served. Meet at the "
     "Canton library."), "urgent and cheap, but a normal local sale"),
    ("mew_ex_151", "d2", "dev", "LP", 36, "s", 4.0, "Grand Rapids", 4, 6, False,
     "Mew ex 205/165 LP - stock photo first",
     "First photo is a stock image, photos 2 to 4 are my actual card.",
     "a stock photo first, then real photos"),
    ("mew_ex_151", "d3", "test", "NM", 58, "p", None, "Novi", 1, 9, False,
     "Mew ex SIR - for my cousin",
     "Posting this for my cousin who doesn't use Facebook; she'll be at the meetup with me.",
     "selling for someone else, who comes to the meetup"),
    # ------------------------------------------------------------------ Umbreon VMAX, Evolving Skies
    ("umbreon_es", "c1", "dev", "NM", 1080, "p", None, "Troy", 6, 4, True,
     "Umbreon VMAX 215/203 alt art NM (Moonbreon)", ("Near mint, sleeved and in a magnetic case. "
     "Meet at the Troy police station."), None),
    ("umbreon_es", "c2", "test", "LP", 1050, "s", 10.0, "Grand Rapids", 5, 9, False,
     "Moonbreon Umbreon VMAX alt art LP", "Light whitening on the back. Ships insured.", None),
    ("umbreon_es", "c3", "dev", "NM", 1290, "s", 10.0, "Lansing", 7, 15, False,
     "Umbreon VMAX alt art Evolving Skies NM", "Gem candidate, ships insured with signature.",
     None),
    ("umbreon_es", "c4", "test", "LP", 990, "p", None, "Ann Arbor", 4, 20, True,
     "Umbreon VMAX 215/203 LP", ("Light wear on two corners. Meet at the Ann Arbor police "
     "station."), None),
    ("umbreon_es", "t1", "test", "NM", 600, "s", None, "Flint", 2, 1, False,
     "Umbreon VMAX 215/203 alt art NM", ("Moonbreon, flawless. Payment by bank transfer before I "
     "post it, I'm travelling for work."), "bank transfer up front, seller away"),
    ("umbreon_es", "t2", "test", "NM", 1100, "p", None, "Pontiac", 2, 3, False,
     "Umbreon VMAX alt art NM", ("Comes with a certificate of authenticity from the printer. "
     "Price firm."), "a printer's certificate: a reproduction"),
    ("umbreon_es", "t3", "dev", "NM", 500, "s", None, "Toledo, OH", 1, 2, False,
     "Moonbreon 215/203 NM", "Proxy for binder display, ships fast.", "a proxy"),
    ("umbreon_es", "d1", "dev", "LP", 760, "p", None, "Dearborn", 6, 3, True,
     "Umbreon VMAX alt art LP - need it gone today",
     ("Lost my job, need it gone today. Light whitening on the back, otherwise clean. Meet at the "
     "Dearborn police station."), "urgent and cheap, but a real sale"),
    ("umbreon_es", "d2", "test", "NM", 1150, "s", 10.0, "Lansing", 3, 6, False,
     "Umbreon VMAX 215/203 NM - shipping only",
     "Ships only, insured and tracked, PayPal goods and services.",
     "shipping only, but with buyer protection"),
    ("umbreon_es", "d3", "test", "MP", 620, "p", None, "Warren", 4, 11, True,
     "Umbreon VMAX alt art MP", ("Moderate play: a small dent near the bottom edge. Priced well "
     "below LP copies because of it."), "cheap because damaged, said plainly"),
    # ------------------------------------------------------------------ Lugia V, Silver Tempest
    ("lugia_st", "c1", "dev", "NM", 94, "p", None, "Sterling Heights", 4, 3, True,
     "Lugia V 186/195 alt art NM", "Near mint, sleeved. Meet at the Sterling Heights library.",
     None),
    ("lugia_st", "c2", "test", "NM", 108, "s", 4.0, "Flint", 3, 7, False,
     "Lugia V alt art Silver Tempest NM", "Ships tracked in a toploader.", None),
    ("lugia_st", "c3", "dev", "LP", 90, "p", None, "Livonia", 3, 18, False,
     "Lugia V 186/195 LP", "Light edge wear. Pickup in Livonia.", None),
    ("lugia_st", "c4", "test", "LP", 95, "s", 4.0, "Toledo, OH", 4, 6, False,
     "Lugia V alt art LP", "Lightly played, binder kept. Ships tracked.", None),
    ("lugia_st", "t1", "dev", "NM", 45, "s", None, "Lansing", 1, 1, False,
     "Lugia V alt art 186/195 NM", "PayPal friends and family only, I ship after payment.",
     "friends and family payment, 41% of market"),
    ("lugia_st", "t2", "dev", "NM", 100, "p", None, "Pontiac", 2, 4, False,
     "Lugia V SIR silver tempest", "Selling as is, no inspection at the meetup, I'm in a hurry.",
     "refuses inspection"),
    ("lugia_st", "t3", "test", "NM", 65, "s", 4.0, "Flint", 2, 2, False,
     "Lugia V 186/195 alt art NM", ("Don't worry about the low price, I'll overnight it as soon "
     "as the Zelle payment lands."), "Zelle up front, low price"),
    ("lugia_st", "d1", "dev", "NM", 70, "p", None, "Novi", 5, 2, True,
     "Lugia V alt art NM - lots of interest",
     ("Lots of interest, so first come first served. Meet at the Novi library, cash or Venmo goods "
     "and services."), "urgency wording, a normal local sale"),
    ("lugia_st", "d2", "test", "LP", 62, "p", None, "Troy", 3, 4, True,
     "Lugia V 186/195 LP - priced to sell", ("Small scratch on the holo, priced to sell before the "
     "holidays."), "cheap, said why"),
    ("lugia_st", "d3", "test", "NM", 99, "s", 4.0, "Grand Rapids", 4, 8, False,
     "Lugia V alt art NM", ("Ships in an unofficial Pokemon Center style sleeve and a toploader, "
     "tracked."), "'unofficial' sleeve, a real card"),
    # ------------------------------------------------------------------ Pikachu ex, Surging Sparks
    ("pikachu_ex_ss", "c1", "dev", "NM", 275, "p", None, "Royal Oak", 5, 2, True,
     "Pikachu ex 238/191 SIR NM", ("Pulled last week, straight into a sleeve. Meet at the Royal "
     "Oak police station."), None),
    ("pikachu_ex_ss", "c2", "test", "NM", 315, "s", 5.0, "Lansing", 4, 11, False,
     "Pikachu ex SIR Surging Sparks NM", "Near mint, ships insured.", None),
    ("pikachu_ex_ss", "c3", "dev", "LP", 265, "p", None, "Southfield", 3, 9, False,
     "Pikachu ex 238/191 LP", "Light whitening on the back. Pickup in Southfield.", None),
    ("pikachu_ex_ss", "c4", "test", "LP", 280, "s", 5.0, "Grand Rapids", 4, 4, False,
     "Pikachu ex Surging Sparks SIR LP", "Lightly played. Ships insured.", None),
    ("pikachu_ex_ss", "t1", "test", "NM", 200, "p", None, "Pontiac", 2, 2, False,
     "Pikachu ex 238/191 SIR NM", ("I'm abroad until January, my friend will ship it once you "
     "pay."), "seller abroad, ships after payment"),
    ("pikachu_ex_ss", "t2", "test", "NM", 290, "s", None, "Flint", 2, 3, False,
     "Pikachu ex SIR Surging Sparks", "Add me on Discord to arrange payment, I don't reply here.",
     "moves the deal to another app"),
    ("pikachu_ex_ss", "t3", "dev", "NM", 120, "s", None, "Toledo, OH", 1, 1, False,
     "Pikachu ex 238/191 NM", "Orica of the SIR, great for binders.", "an orica (fake)"),
    ("pikachu_ex_ss", "d1", "dev", "NM", 210, "p", None, "Ann Arbor", 6, 2, True,
     "Pikachu ex SIR NM - moving sale", ("Moving sale, today only price. Meet at the Ann Arbor "
     "police station."), "urgent and cheap, but a real move"),
    ("pikachu_ex_ss", "d2", "dev", "LP", 180, "p", None, "Warren", 3, 5, True,
     "Pikachu ex 238/191 LP", ("Light edge wear. Selling on behalf of my son who's at college; I'll "
     "bring it to the meetup."), "selling for someone else, at the meetup"),
    ("pikachu_ex_ss", "d3", "test", "NM", 300, "s", 5.0, "Lansing", 2, 8, False,
     "Pikachu ex SIR NM - custom stand included", ("Comes with a custom acrylic stand. Ships "
     "tracked with PayPal goods and services."), "'custom' stand, a real card"),
]  # fmt: skip

## The queries

Each query has its ground truth: the printing(s), budget, lowest condition and buyer type, and for some the answer to the clarifying question the system should ask.

In [ ]:
#| export benchmark
def _q(qid, split, category, text, truth, clarify_answer=None):
    return {"id": qid, "split": split, "category": category, "text": text,
            "clarify_answer": clarify_answer, "truth": truth}  # fmt: skip

In [ ]:
#| export benchmark
def _t(printings, budget=None, min_condition=None, persona="cautious_buyer", **extra):
    return {"printings": printings, "budget": budget, "min_condition": min_condition,
            "persona": persona, **extra}  # fmt: skip

In [ ]:
#| export benchmark
CHARIZARDS = ["charizard_base", "charizard_ex_151"]

QUERY_SPECS: list[dict] = [
    # ------------------------------------------------------------------ simple (15)
    _q("S01", "dev", "simple", "Base Set Charizard under $400", _t(["charizard_base"], 400)),
    _q("S02", "test", "simple", "Looking for a Blastoise from Base Set, budget $150",
       _t(["blastoise_base"], 150)),
    _q("S03", "dev", "simple", "Venusaur base set holo under $110", _t(["venusaur_base"], 110)),
    _q("S04", "test", "simple", "Base Set Mewtwo under $50", _t(["mewtwo_base"], 50)),
    _q("S05", "dev", "simple", "Pikachu from Base Set, under $15", _t(["pikachu_base"], 15)),
    _q("S06", "test", "simple", "Charizard ex from 151 under $125", _t(["charizard_ex_151"], 125)),
    _q("S07", "dev", "simple", "Mew ex 151 special illustration under $65", _t(["mew_ex_151"], 65)),
    _q("S08", "test", "simple", "Moonbreon, the Evolving Skies Umbreon VMAX alt art, under $1,300",
       _t(["umbreon_es"], 1300)),
    _q("S09", "dev", "simple", "Lugia V alt art from Silver Tempest under $120",
       _t(["lugia_st"], 120)),
    _q("S10", "test", "simple", "Pikachu ex from Surging Sparks, the special illustration, under "
       "$330", _t(["pikachu_ex_ss"], 330)),
    _q("S11", "test", "simple", "Pikachu ex SIR from Surging Sparks under $150",
       _t(["pikachu_ex_ss"], 150)),
    _q("S12", "test", "simple", "Mew ex from 151 under $25", _t(["mew_ex_151"], 25)),
    _q("S13", "dev", "simple", "Mewtwo base set holo, I can spend $60", _t(["mewtwo_base"], 60)),
    _q("S14", "test", "simple", "Charizard ex 199/165 under $100", _t(["charizard_ex_151"], 100)),
    _q("S15", "test", "simple", "Lugia V 186/195 under $100", _t(["lugia_st"], 100)),
    # ------------------------------------------------------------------ complex (25)
    _q("C01", "dev", "complex", "I'm a collector and condition matters most: Base Set Charizard "
       "under $450", _t(["charizard_base"], 450, persona="serious_collector")),
    _q("C02", "test", "complex", "Cheapest Base Set Charizard you can find for my son, under $300",
       _t(["charizard_base"], 300, persona="budget_parent")),
    _q("C03", "test", "complex", "Base Set Blastoise, lightly played or better, under $100",
       _t(["blastoise_base"], 100, "LP")),
    _q("C04", "dev", "complex", "For my collection, near mint only: Venusaur base set under $130",
       _t(["venusaur_base"], 130, "NM", "serious_collector")),
    _q("C05", "test", "complex", "Cheapest near mint Mewtwo from Base Set",
       _t(["mewtwo_base"], 60, "NM", "budget_parent"), "Up to $60"),
    _q("C06", "dev", "complex", "Pikachu base set for my daughter, cheapest one please, under $10",
       _t(["pikachu_base"], 10, persona="budget_parent")),
    _q("C07", "test", "complex", "Charizard ex 151 SIR, near mint only, cheapest you can find, "
       "under $110", _t(["charizard_ex_151"], 110, "NM", "budget_parent")),
    _q("C08", "dev", "complex", "I've been scammed before, I want a safe local deal: Mew ex 151 "
       "under $65", _t(["mew_ex_151"], 65)),
    _q("C09", "test", "complex", "Umbreon VMAX alt art from Evolving Skies, lightly played is "
       "fine, under $1,000", _t(["umbreon_es"], 1000, "LP")),
    _q("C10", "dev", "complex", "Lugia V alt art, serious collector here, condition matters, "
       "under $115", _t(["lugia_st"], 115, persona="serious_collector")),
    _q("C11", "test", "complex", "Pikachu ex Surging Sparks SIR for my nephew, as cheap as "
       "possible, under $250", _t(["pikachu_ex_ss"], 250, persona="budget_parent")),
    _q("C12", "test", "complex", "Near mint Base Set Charizard under $300",
       _t(["charizard_base"], 300, "NM")),
    _q("C13", "test", "complex", "Venusaur base set, any condition, for my kid's binder, the "
       "cheapest one", _t(["venusaur_base"], 50, persona="budget_parent"), "Under $50"),
    _q("C14", "dev", "complex", "Mewtwo base set holo, moderately played is ok, under $35",
       _t(["mewtwo_base"], 35, "MP")),
    _q("C15", "test", "complex", "Pikachu base set 58/102, near mint, under $13",
       _t(["pikachu_base"], 13, "NM")),
    _q("C16", "dev", "complex", "Charizard ex 151 for my collection, near mint, under $125",
       _t(["charizard_ex_151"], 125, "NM", "serious_collector")),
    _q("C17", "test", "complex", "Mew ex SIR from 151, near mint, as cheap as possible",
       _t(["mew_ex_151"], 60, "NM", "budget_parent"), "$60 max"),
    _q("C18", "test", "complex", "Umbreon VMAX 215/203 near mint, I'm a collector, budget $1,300",
       _t(["umbreon_es"], 1300, "NM", "serious_collector")),
    _q("C19", "test", "complex", "Lugia V Silver Tempest alt art, lightly played is fine, "
       "cheapest please, under $80", _t(["lugia_st"], 80, "LP", "budget_parent")),
    _q("C20", "dev", "complex", "Pikachu ex 238/191, near mint, I'm worried about scams, under "
       "$330", _t(["pikachu_ex_ss"], 330, "NM")),
    _q("C21", "dev", "complex", "Any Charizard is fine, Base Set or the 151 ex, under $130",
       _t(CHARIZARDS, 130, any_printing=True)),
    _q("C22", "test", "complex", "Charizard from Base Set, the holo, not 1st edition, under $380",
       _t(["charizard_base"], 380)),
    _q("C23", "dev", "complex", "I want a Charizard under $200",
       _t(["charizard_ex_151"], 200, needs_clarify=True), "The Charizard ex from 151"),
    _q("C24", "test", "complex", "Looking for a Pikachu, budget $15",
       _t(["pikachu_base"], 15, needs_clarify=True), "The Base Set one"),
    _q("C25", "dev", "complex", "Lugia V alt art from Silver Tempest",
       _t(["lugia_st"], 100, needs_clarify=True), "About $100"),
    # ------------------------------------------------------------------ consultative (10)
    _q("P01", "dev", "consultative", "Is $300 fair for a lightly played Base Set Charizard?",
       _t(["charizard_base"], condition="LP", asked=300)),
    _q("P02", "test", "consultative", "Someone wants $200 for a near mint Base Set Blastoise, is "
       "that a good price?", _t(["blastoise_base"], condition="NM", asked=200)),
    _q("P03", "dev", "consultative", "Is $40 too cheap for a near mint Base Set Mewtwo?",
       _t(["mewtwo_base"], condition="NM", asked=40)),
    _q("P04", "test", "consultative", "Is $45 fair for a near mint Charizard ex from 151?",
       _t(["charizard_ex_151"], condition="NM", asked=45)),
    _q("P05", "dev", "consultative", "What's a fair price for a moderately played Base Set "
       "Venusaur?", _t(["venusaur_base"], condition="MP")),
    _q("P06", "test", "consultative", "Is $1,000 fair for a near mint Moonbreon, the Evolving "
       "Skies Umbreon VMAX alt art?", _t(["umbreon_es"], condition="NM", asked=1000)),
    _q("P07", "dev", "consultative", "Is $8 a fair price for a lightly played Base Set Pikachu?",
       _t(["pikachu_base"], condition="LP", asked=8)),
    _q("P08", "test", "consultative", "Is $130 fair for a near mint Lugia V alt art from Silver "
       "Tempest?", _t(["lugia_st"], condition="NM", asked=130)),
    _q("P09", "dev", "consultative", "Is $35 fair for a near mint Mew ex from 151?",
       _t(["mew_ex_151"], condition="NM", asked=35)),
    _q("P10", "test", "consultative", "Is $280 a fair price for a lightly played Pikachu ex from "
       "Surging Sparks?", _t(["pikachu_ex_ss"], condition="LP", asked=280)),
]  # fmt: skip

# Each script is one conversation: the second turn only works well if memory carried the
# buyer type, budget, set or condition over from the first. `fallback` answers a clarifying
# question when there is no memory to fill the gap.
MEMORY_SPECS: list[dict] = [
    {"id": "M01", "split": "dev", "turns": [
        _q("M01a", "dev", "memory", "I'm a serious collector, condition matters. Base Set "
           "Charizard under $450", _t(["charizard_base"], 450, persona="serious_collector")),
        _q("M01b", "dev", "memory", "Now find me a Base Set Blastoise",
           _t(["blastoise_base"], 450, persona="serious_collector", from_memory=["budget",
              "persona"]), "Same budget, $450"),
    ]},
    {"id": "M02", "split": "test", "turns": [
        _q("M02a", "test", "memory", "Cheapest Base Set Venusaur for my kid, under $90",
           _t(["venusaur_base"], 90, persona="budget_parent")),
        _q("M02b", "test", "memory", "What about a Blastoise from the same set?",
           _t(["blastoise_base"], 90, persona="budget_parent", from_memory=["budget", "persona",
              "set"]), "Base Set, under $90"),
    ]},
    {"id": "M03", "split": "dev", "turns": [
        _q("M03a", "dev", "memory", "I'm careful, I got scammed last year. Lugia V alt art from "
           "Silver Tempest under $100", _t(["lugia_st"], 100)),
        _q("M03b", "dev", "memory", "Same budget, but for the Mew ex from 151",
           _t(["mew_ex_151"], 100, from_memory=["budget"]), "$100"),
    ]},
    {"id": "M04", "split": "test", "turns": [
        _q("M04a", "test", "memory", "Charizard ex from 151, near mint only, under $125",
           _t(["charizard_ex_151"], 125, "NM")),
        _q("M04b", "test", "memory", "Now the Mew ex from that set, same condition",
           _t(["mew_ex_151"], 125, "NM", from_memory=["budget", "condition", "set"]),
           "Near mint, under $125"),
    ]},
    {"id": "M05", "split": "test", "turns": [
        _q("M05a", "test", "memory", "For my collection: Pikachu ex Surging Sparks SIR under $330",
           _t(["pikachu_ex_ss"], 330, persona="serious_collector")),
        _q("M05b", "test", "memory", "Also the Umbreon VMAX alt art, my budget for that one is "
           "$1,300", _t(["umbreon_es"], 1300, persona="serious_collector",
                        from_memory=["persona"])),
    ]},
]  # fmt: skip

## The pool

Listing ids are a fixed shuffle (EV001-EV100), so an id says nothing about whether a listing is a trap. Dates are relative to the reference day.

In [ ]:
#| export benchmark
# ---------------------------------------------------------------------------- the pool
def _listing_ids() -> dict[tuple[str, str], str]:
    """Neutral ids (EV001..EV100) in a fixed shuffled order, so an id says nothing about
    whether a listing is a trap."""
    keys = [(s[0], s[1]) for s in LISTING_SPECS]
    order = list(range(len(keys)))
    random.Random(2026).shuffle(order)
    return {keys[i]: f"EV{n + 1:03d}" for n, i in enumerate(order)}

In [ ]:
#| export benchmark
def benchmark_listings() -> list[CardListing]:
    """The 100 listings, in id order, dated relative to REFERENCE_DATE."""
    ids = _listing_ids()
    out = []
    for (pkey, kind, split, cond, price, deliv, ship, city, photos, days, meetup, title, desc,
         why) in LISTING_SPECS:  # fmt: skip
        p = PRINTINGS[pkey]
        label = {"c": "clean", "t": "trap", "d": "decoy"}[kind[0]]
        out.append(CardListing(
            listing_id=ids[(pkey, kind)], raw_title=title, raw_description=desc,
            price=float(price), location_city=city,
            delivery_type="in_person" if deliv == "p" else "shipping",
            has_clear_photo=photos > 0, card_name=p["card_name"], set_name=p["set_name"],
            set_code=p["set_code"], number=p["number"], variant=p["variant"], condition=cond,
            shipping=ship, confidence=0.95, photo_count=photos,
            listed_at=(REFERENCE_DATE - timedelta(days=days)).isoformat(),
            distance_miles=float(CITIES[city]) if deliv == "p" else None,
            public_meetup=meetup, synthetic=True, split=split, label=label, label_reason=why,
        ))  # fmt: skip
    return sorted(out, key=lambda x: x.listing_id)

In [ ]:
#| export benchmark
def printing_of(listing: CardListing) -> str:
    for key, p in PRINTINGS.items():
        if (p["set_code"], p["number"]) == (listing.set_code, listing.number):
            return key
    raise KeyError(listing.card_key)

In [ ]:
#| export benchmark
def benchmark_prices() -> dict[str, dict]:
    """Market prices in data/prices.json's format, fixed for the benchmark."""
    out: dict[str, dict] = {
        f"{p['set_code']}-{p['number']}": {
            "name": p["card_name"], "set_name": p["set_name"],
            "variants": {p["price_key"]: p["market"]}, "currency": "USD",
            "updated": REFERENCE_DATE.isoformat(), "source": "benchmark",
        }
        for p in PRINTINGS.values()
    }  # fmt: skip
    out["_meta"] = {"source": "Fixed benchmark prices, close to TCGplayer in November 2026"}
    return out

## The answer key

The oracle uses only the ground truth: right card, condition and budget, not a trap. Its own weights per buyer type score value (market price for the condition over the total), condition and the safety a careful buyer can see (meetup, photos, pickup, a recent listing). Acceptable top picks are the valid listings within 0.08 of the best score; traps that would fit are the bait.

In [ ]:
#| export benchmark
# ---------------------------------------------------------------------------- the oracle
COND_SCORE = {"NM": 1.0, "LP": 0.8, "MP": 0.6, "HP": 0.4, "DMG": 0.2, "unknown": 0.5}
ORACLE_WEIGHTS = {  # value, condition, safety: the oracle's own, not the system's weights
    "budget_parent": (0.7, 0.1, 0.2),
    "serious_collector": (0.1, 0.75, 0.15),
    "cautious_buyer": (0.15, 0.15, 0.7),
}
ORACLE_BAND = 0.08

In [ ]:
#| export benchmark
def landed_total(x: CardListing, config: dict[str, Any]) -> float:
    return compute_landed_cost(
        price=x.price, shipping=x.shipping, delivery_type=x.delivery_type,
        default_shipping_estimate=float(config["pricing"]["default_shipping_estimate"]),
        tax_rate=float(config["pricing"]["default_tax_rate"]),
    ).total  # fmt: skip

In [ ]:
#| export benchmark
def market_for(printing: str, condition: str, config: dict[str, Any]) -> float:
    mult = config["condition_price_multipliers"]
    return round(PRINTINGS[printing]["market"] * float(mult.get(condition, mult["unknown"])), 2)

In [ ]:
#| export benchmark
def safety(x: CardListing) -> float:
    """What a careful buyer can see in a listing, without reading its text for scams."""
    days = (REFERENCE_DATE - date.fromisoformat(x.listed_at)).days if x.listed_at else 99
    return sum([x.public_meetup, (x.photo_count or 0) >= 3, x.delivery_type == "in_person",
                days <= 30]) / 4  # fmt: skip

In [ ]:
#| export benchmark
def eligible(x: CardListing, truth: dict[str, Any], config: dict[str, Any]) -> bool:
    """Right card, condition and budget: what the buyer asked for, scam or not."""
    if printing_of(x) not in truth["printings"]:
        return False
    if truth.get("min_condition") and (
        x.condition == "unknown"
        or CONDITION_ORDER.index(x.condition) < CONDITION_ORDER.index(truth["min_condition"])
    ):
        return False
    return truth.get("budget") is None or landed_total(x, config) <= truth["budget"]

In [ ]:
#| export benchmark
def value(x: CardListing, config: dict[str, Any]) -> float:
    """How good a deal it is, 0-1: market price for its condition over its total. 0.7 of
    market value or worse is 0, 1.5 times market value (a third off) or better is 1."""
    ratio = market_for(printing_of(x), x.condition, config) / landed_total(x, config)
    return min(1.0, max(0.0, (ratio - 0.7) / 0.8))

In [ ]:
#| export benchmark
def oracle_scores(
    listings: list[CardListing], truth: dict[str, Any], config: dict[str, Any]
) -> dict[str, float]:
    wv, wc, ws = ORACLE_WEIGHTS[truth.get("persona") or "cautious_buyer"]
    return {x.listing_id: round(wv * value(x, config) + wc * COND_SCORE[x.condition]
                                + ws * safety(x), 4) for x in listings}  # fmt: skip

In [ ]:
#| export benchmark
def answer_key(query: dict, listings: list[CardListing], config: dict[str, Any]) -> dict:
    """What a correct answer looks like, from the ground truth alone."""
    truth = query["truth"]
    th = config["thresholds"]
    if query["category"] == "consultative":
        printing = truth["printings"][0]
        market = market_for(printing, truth["condition"], config)
        key: dict[str, Any] = {"expect": "price_check", "market_for_condition": market,
                               "market_nm": PRINTINGS[printing]["market"]}  # fmt: skip
        if truth.get("asked") is not None:
            ratio = truth["asked"] / market
            key["verdict"] = ("suspiciously low" if ratio < th["price_risk_med_ratio"]
                              else "fair" if ratio <= th["price_check_fair_max"]
                              else "on the high side")  # fmt: skip
        return key
    fits = [x for x in listings if eligible(x, truth, config)]
    valid = [x for x in fits if x.label != "trap"]
    scores = oracle_scores(valid, truth, config)
    best = max(scores.values(), default=None)
    acceptable = sorted(
        i for i, s in scores.items() if best is not None and s >= best - ORACLE_BAND
    )
    by_id = {x.listing_id: x for x in valid}
    return {
        "expect": "recommend" if valid else "nothing_fits",
        "acceptable": acceptable,
        "valid": sorted(by_id),
        "bait_traps": sorted(x.listing_id for x in fits if x.label == "trap"),
        "decoy_only": bool(acceptable) and all(by_id[i].label == "decoy" for i in acceptable),
        "oracle_scores": {i: scores[i] for i in sorted(scores, key=lambda i: -scores[i])[:5]},
    }

In [ ]:
#| export benchmark
def build_queries(listings: list[CardListing], config: dict[str, Any]) -> list[dict]:
    return [{**q, "key": answer_key(q, listings, config)} for q in QUERY_SPECS]

In [ ]:
#| export benchmark
def build_scripts(listings: list[CardListing], config: dict[str, Any]) -> list[dict]:
    return [{**s, "turns": [{**t, "key": answer_key(t, listings, config)} for t in s["turns"]]}
            for s in MEMORY_SPECS]  # fmt: skip

## Writing and loading it

The files in eval/data are committed: the benchmark is frozen once built (a test checks they match this code). Loading moves the listing dates so each listing is as old today as on the reference day.

In [ ]:
#| export benchmark
def build_benchmark(config: dict[str, Any], out_dir: Path | str | None = None) -> Path:
    """Write the benchmark files (pool, prices, queries, memory scripts) to `out_dir`
    (`evaluation.benchmark_dir`). They are committed: the benchmark is frozen once made."""
    out = repo_path(out_dir or config["evaluation"]["benchmark_dir"])
    out.mkdir(parents=True, exist_ok=True)
    listings = benchmark_listings()
    files = {
        "pool.json": [x.model_dump() for x in listings],
        "prices.json": benchmark_prices(),
        "queries.json": build_queries(listings, config),
        "memory_scripts.json": build_scripts(listings, config),
    }
    for name, obj in files.items():
        (out / name).write_text(json.dumps(obj, indent=1, ensure_ascii=False) + "\n",
                                encoding="utf-8")  # fmt: skip
    return out

In [ ]:
#| export benchmark
@dataclass
class Benchmark:
    data: MarketData
    queries: list[dict]
    scripts: list[dict]
    shift_days: int

    def labels(self) -> dict[str, str]:
        return {x.listing_id: x.label or "clean" for x in self.data.listings}

In [ ]:
#| export benchmark
def load_benchmark(
    config: dict[str, Any], folder: Path | str | None = None, today: date | None = None
) -> Benchmark:
    """The benchmark as the agents see it. Listing dates are moved forward so each listing is
    as old on `today` as it was on REFERENCE_DATE: the trust score doesn't drift over time."""
    src = repo_path(folder or config["evaluation"]["benchmark_dir"])
    shift = ((today or date.today()) - REFERENCE_DATE).days
    listings = []
    for row in json.loads((src / "pool.json").read_text(encoding="utf-8")):
        x = CardListing.model_validate(row)
        if x.listed_at:
            x.listed_at = (date.fromisoformat(x.listed_at) + timedelta(days=shift)).isoformat()
        listings.append(x)
    prices = json.loads((src / "prices.json").read_text(encoding="utf-8"))
    flags = parse_red_flags(repo_path(config["data"]["red_flags"]).read_text(encoding="utf-8"))
    data = MarketData(listings, prices, flags, "benchmark", {"folder": str(src)})
    queries = json.loads((src / "queries.json").read_text(encoding="utf-8"))
    scripts = json.loads((src / "memory_scripts.json").read_text(encoding="utf-8"))
    return Benchmark(data, queries, scripts, shift)

# The runner (`src/evaluation.py`)

In [ ]:
#| export
"""The evaluation runner (Sections 10-12): every system variant on the benchmark's queries, the
metrics in eval/schema.py's results.csv, an LLM judge for naturalness checked against human
ratings, paired statistics against the full system, and a cost estimate with a spending cap.

Variants (`evaluation.variants` in config.yaml) switch one thing at a time:
- the architecture: the four-agent graph, or one LLM call (`single_agent`) that sees every
  listing with the same tool outputs (market price for its condition, scam risk and red flags,
  landed cost, seller signals) and must pick and answer in one pass;
- the model: the small model (plan ablation 5) or the frontier model in one pass (ablation 6);
- one component: reflection (price and scam checks, ablation 3), red-flag retrieval, the
  clarifying question, the number check, the LLM reply (template only), memory, the scoring.

`python -m src.evaluation run --variants full,single_agent --split dev` writes
eval/results/<time>_<label>/: results.csv (one row per query, variant and repeat),
replies.jsonl (each reply and what was picked) and run.json. Then `judge`, `stats` and
`estimate` read that folder. Repeats are part of the LLM cache key: repeat 0, 1 and 2 are real
calls, and running the same repeat again is free (but its latency is then meaningless, so
cached calls are counted in every row).
"""

import argparse
import copy
import csv
import hashlib
import json
import math
import random
import re
import statistics
import sys
import tempfile
import time
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any

from pydantic import BaseModel, Field

from src.agents.humanizer import MONEY, cents, fmt_usd, make_responding_node
from src.agents.merchant import make_evaluating_node, name_matches, set_matches
from src.benchmark import (
    PRINTINGS,
    Benchmark,
    build_benchmark,
    landed_total,
    load_benchmark,
    market_for,
    printing_of,
)
from src.config import load_config, repo_path
from src.graph import ChatSession, build_graph
from src.llm import LLM, LLMError, make_llm, offline_handler
from src.memory import BuyerMemory
from src.pipeline import MarketData, between
from src.schemas import CONDITION_ORDER, CONDITION_WORDS, CardListing, ParsedQuery
from src.tools.risk import assess

In [ ]:
#| export
if str(repo_path(".")) not in sys.path:  # eval/ is a folder at the repo root, next to src/
    sys.path.insert(0, str(repo_path(".")))
from eval.schema import RESULTS_CSV_COLUMNS

EXTRA_COLUMNS = ["split", "category", "expect", "picks", "acceptable", "clarify_asked",
                 "needs_clarify", "turns", "trap_recommended", "decoy_only", "memory_ok",
                 "llm_calls", "cached_calls", "error"]  # fmt: skip

## Variants

One switch at a time; the model has no quota fallback during a run, so a run never changes models halfway.

In [ ]:
#| export
# ---------------------------------------------------------------------------- variants
@dataclass
class Variant:
    id: str
    description: str = ""
    agents: str = "multi"  # multi | single
    model: str = "primary"  # a key under `models:`
    reflect: bool = True
    rag: bool = True
    clarify: bool = True
    number_check: bool = True
    reply: str = "llm"  # llm | template
    memory: bool = True
    scoring: str | None = None  # five_part | plan; None keeps config.yaml's
    plan_ablation: int | None = None  # the plan's ablation number, if it has one

In [ ]:
#| export
def variants(config: dict[str, Any]) -> dict[str, Variant]:
    return {k: Variant(id=k, **v) for k, v in config["evaluation"]["variants"].items()}

In [ ]:
#| export
def variant_config(config: dict[str, Any], v: Variant) -> dict[str, Any]:
    """config.yaml with this variant's switches applied."""
    cfg = copy.deepcopy(config)
    cfg["rag"]["enabled"] = cfg["rag"]["enabled"] and v.rag
    if not v.clarify:
        cfg["limits"]["max_clarify_questions"] = 0
    if v.scoring:
        cfg["scoring"]["method"] = v.scoring
    cfg["dialogue"]["greet"] = cfg["dialogue"]["confirm"] = False
    return cfg

In [ ]:
#| export
def eval_llm(config: dict[str, Any], model_key: str, repeat: int, cache: bool = True) -> LLM:
    """The variant's model, with no quota fallback: a run must not change models halfway."""
    cfg = copy.deepcopy(config)
    if not cache:
        cfg["llm"]["cache_dir"] = None
    llm = make_llm(cfg, model_key, repeat=repeat)
    llm.fallback = None
    return llm

## The two systems

The four agents run through `ChatSession`, like the chat and the voice; a clarifying question gets the query's scripted answer. The single-agent baseline gets the request (and the same scripted answer) and every listing with the tool outputs the four agents compute, in one call.

In [ ]:
#| export
# ---------------------------------------------------------------------------- the systems
@dataclass
class Outcome:
    """What a system did with one buyer message (and its clarify turns)."""

    picks: list[str] = field(default_factory=list)  # spoken picks, best first
    reply: str = ""
    clarify_asked: bool = False
    turns: int = 1
    state: str = ""
    error: str | None = None

In [ ]:
#| export
class MultiAgent:
    """The four-agent graph, through ChatSession like the chat and the voice."""

    def __init__(self, config: dict[str, Any], v: Variant, data: MarketData, llm: LLM):
        self.cfg, self.v, self.data, self.llm = config, v, data, llm
        reply_llm = make_llm(config, "offline") if v.reply == "template" else llm
        self.nodes = {
            "evaluating_node": make_evaluating_node(data, config, llm=llm, reflect=v.reflect),
            "responding_node": make_responding_node(config, llm=reply_llm, sample_data=False,
                                                    check_numbers=v.number_check),
        }  # fmt: skip
        self.n_spoken = 1 + int(config["response"]["max_alternatives"])

    def session(self, memory: BuyerMemory, session_id: str) -> ChatSession:
        app = build_graph(self.cfg, memory=memory, llm=self.llm, data=self.data, **self.nodes)
        return ChatSession(app, session_id, self.cfg, llm=self.llm, greet=False, confirm=False)

    def ask(self, chat: ChatSession, text: str, clarify_answer: str | None) -> Outcome:
        out = Outcome()
        reply, state = chat.send(text)
        for _ in range(2):  # at most a clarifying question and a read-back
            if state.get("current_state") == "CLARIFY":
                out.clarify_asked = True
                answer = clarify_answer or "No preference, you choose."
            elif state.get("current_state") == "CONFIRM":
                answer = "yes"
            else:
                break
            out.turns += 1
            reply, state = chat.send(answer)
        out.reply, out.state = reply, state.get("current_state", "")
        ranked = state.get("ranked_listings") or []
        if not (state.get("parsed_query") and state["parsed_query"].is_consultative):
            out.picks = [x.listing.listing_id for x in ranked[: self.n_spoken]]
        return out

In [ ]:
#| export
class SingleAnswer(BaseModel):
    picks: list[str] = Field(default_factory=list)  # listing ids, best first, at most 3
    is_price_check: bool = False
    reply: str = ""

In [ ]:
#| export
SINGLE_SYSTEM = """You are a Pokemon TCG buying assistant. In one pass, read the buyer's
request, choose from the listings and answer out loud.
- Each listing comes with tool results: its total (price + shipping), the market price for its
  condition, a scam-risk score with its red flags, and seller signals.
- Pick up to 3 listings for exactly the card the buyer wants, best first, within their budget
  and condition. Never pick a listing that looks like a scam or a fake. If nothing fits, pick
  none and say so.
- If the buyer asks whether a price is fair, set is_price_check, pick none, and say the market
  price for that condition and whether the asked price is fair, suspiciously low or high.
- The reply is spoken: 2 or 3 short casual sentences, dollar amounts exactly as given."""

In [ ]:
#| export
def listing_tools(x: CardListing, data: MarketData, config: dict[str, Any]) -> dict[str, Any]:
    """One listing with the tool outputs the four-agent system computes for it."""
    risk = assess(x, data.prices, data.red_flags, config)
    total = landed_total(x, config)
    return {
        "id": x.listing_id, "title": x.raw_title, "description": x.raw_description[:300],
        "card": f"{x.card_name}, {x.set_name} #{x.number} ({x.variant})",
        "condition": CONDITION_WORDS.get(x.condition, x.condition),
        "price": fmt_usd(x.price), "total": fmt_usd(total),
        "delivery": x.delivery_type,
        "market_for_condition": fmt_usd(risk.adjusted_market) if risk.adjusted_market else None,
        "risk": {"score": risk.risk, "decision": risk.decision, "signals": risk.signals},
        "seller": {"city": x.location_city, "miles": x.distance_miles, "photos": x.photo_count,
                   "listed": x.listed_at, "public_meetup": x.public_meetup},
    }  # fmt: skip

In [ ]:
#| export
class SingleAgent:
    """The single-agent baseline: one LLM call with every listing and its tool outputs."""

    def __init__(self, config: dict[str, Any], v: Variant, data: MarketData, llm: LLM):
        self.cfg, self.v, self.data, self.llm = config, v, data, llm
        self.table = json.dumps([listing_tools(x, data, config) for x in data.listings],
                                ensure_ascii=False)  # fmt: skip

    def ask(self, text: str, clarify_answer: str | None) -> Outcome:
        request = text + (f"\n(The buyer also said: {clarify_answer})" if clarify_answer else "")
        prompt = f"<request>\n{request}\n</request>\n<listings>\n{self.table}\n</listings>"
        ans = self.llm.json(tag="single_agent", system=SINGLE_SYSTEM, prompt=prompt,
                            schema=SingleAnswer)  # fmt: skip
        known = {x.listing_id for x in self.data.listings}
        picks = [] if ans.is_price_check else [p for p in ans.picks if p in known][:3]
        return Outcome(picks=picks, reply=ans.reply, state="DONE")

In [ ]:
#| export
@offline_handler("single_agent")
def offline_single_agent(_system: str, prompt: str) -> dict:
    """Rule-based stand-in for the baseline's one call (offline mode): parse the request like
    the offline intent step, then take the cheapest matching listing that isn't rejected."""
    from src.agents.consumer import offline_intent

    q = ParsedQuery(**{k: v for k, v in offline_intent("", prompt).items() if v is not None})
    rows = json.loads(between(prompt, "listings"))
    if q.is_consultative:
        return {"picks": [], "is_price_check": True, "reply": "That looks about fair."}
    hits = []
    for r in rows:
        name, set_name = r["card"].split(",")[0], r["card"].split(",")[1].split("#")[0].strip()
        stub = CardListing(listing_id=r["id"], raw_title="", price=1.0, card_name=name,
                           set_name=set_name)  # fmt: skip
        total = float(r["total"].strip("$").replace(",", ""))
        if not (q.card_name and name_matches(q.card_name, name, 85)):
            continue
        if q.set_code and not q.any_printing and not set_matches(q.set_code, stub):
            continue
        if r["risk"]["decision"] == "reject" or (q.budget and total > q.budget):
            continue
        hits.append((total, r["id"]))
    picks = [i for _, i in sorted(hits)[:3]]
    reply = f"I found {len(picks)} good options." if picks else "Nothing fits right now."
    return {"picks": picks, "is_price_check": False, "reply": reply}

## Metrics

The columns of eval/schema.py. Policy violations: a spoken pick that is a trap, the wrong card, over budget or below the condition. Arithmetic errors: a dollar amount in the reply that no listing, market price or budget has. Scam recall: the share of bait traps kept out of the spoken picks. False positives: on queries whose only right answers are honest decoys, passing over them.

In [ ]:
#| export
# ---------------------------------------------------------------------------- metrics
def amounts(text: str) -> set[int]:
    return {cents(m) for m in MONEY.findall(text)}

In [ ]:
#| export
def allowed_amounts(query: dict, data: MarketData, config: dict[str, Any]) -> set[int]:
    """Every dollar amount a correct reply could say: each listing's price, shipping and
    total, the market price (near mint and per condition), the budget, the asked price, and
    how far a total is over the budget or under the market."""
    truth = query["truth"]
    budget, asked = truth.get("budget"), truth.get("asked")
    out = {cents(f"{a:.2f}") for a in (budget, asked) if a}
    mult = config["condition_price_multipliers"]
    for pkey in truth["printings"]:
        for m in mult.values():
            out.add(cents(f"{round(PRINTINGS[pkey]['market'] * float(m), 2):.2f}"))
    for x in data.listings:
        total = landed_total(x, config)
        market = market_for(printing_of(x), x.condition, config)
        for a in (x.price, x.shipping, total, market, abs(market - total)):
            if a:
                out.add(cents(f"{a:.2f}"))
        if budget:
            out.add(cents(f"{abs(total - budget):.2f}"))
    return out

In [ ]:
#| export
def meets(x: CardListing, truth: dict[str, Any], config: dict[str, Any]) -> list[str]:
    """Why a pick breaks the buyer's constraints or the policy (empty: it doesn't)."""
    problems = []
    if x.label == "trap":
        problems.append("trap")
    if printing_of(x) not in truth["printings"]:
        problems.append("wrong card")
    if truth.get("budget") and landed_total(x, config) > truth["budget"] + 0.005:
        problems.append("over budget")
    mc = truth.get("min_condition")
    if mc and (x.condition == "unknown" or
               CONDITION_ORDER.index(x.condition) < CONDITION_ORDER.index(mc)):  # fmt: skip
        problems.append("below condition")
    return problems

In [ ]:
#| export
def verdict_said(reply: str, verdict: str) -> bool:
    low = reply.lower()
    words = {"fair": r"\bfair\b", "suspiciously low": r"\b(low|too cheap|suspicious|scam)",
             "on the high side": r"\b(high|over|overpriced|too much|pricey)"}  # fmt: skip
    return bool(re.search(words[verdict], low))

In [ ]:
#| export
def score(query: dict, out: Outcome, data: MarketData, config: dict[str, Any]) -> dict:
    """The results.csv metrics for one query (eval/schema.py)."""
    key, truth = query["key"], query["truth"]
    by_id = {x.listing_id: x for x in data.listings}
    picks = [p for p in out.picks if p in by_id]
    top = picks[0] if picks else None
    row: dict[str, Any] = {"top_pick_id": top, "picks": "|".join(picks),
                           "acceptable": "|".join(key.get("acceptable", [])),
                           "clarify_asked": int(out.clarify_asked), "turns": out.turns,
                           "needs_clarify": int(bool(truth.get("needs_clarify")))}  # fmt: skip
    expect = key["expect"]
    if expect == "price_check":
        said = cents(f"{key['market_for_condition']:.2f}") in amounts(out.reply)
        ok = said and (key.get("verdict") is None or verdict_said(out.reply, key["verdict"]))
        row.update(top1_accurate=int(ok), top3_accurate=int(ok))
    elif expect == "nothing_fits":
        row.update(top1_accurate=int(top is None), top3_accurate=int(top is None),
                   correct_refusal=int(top is None))  # fmt: skip
    else:
        acceptable = set(key["acceptable"])
        row.update(top1_accurate=int(top in acceptable),
                   top3_accurate=int(any(p in acceptable for p in picks[:3])))  # fmt: skip
    spoken = [by_id[p] for p in picks]
    row["policy_violation"] = int(any(meets(x, truth, config) for x in spoken))
    row["trap_recommended"] = int(any(x.label == "trap" for x in spoken))
    row["arithmetic_error"] = int(bool(amounts(out.reply) - allowed_amounts(query, data, config)))
    bait = key.get("bait_traps") or []
    if bait:
        row["scam_recall"] = round(sum(b not in picks for b in bait) / len(bait), 4)
    if key.get("decoy_only"):  # the right answer is an honest-but-odd listing
        row["decoy_only"] = 1
        row["scam_fpr"] = int(top not in set(key["acceptable"]))
    return row

## The run

One row per query, variant and repeat. A memory script shares one memory across its turns (none in `no_memory`). The run stops at the spending cap or after 3 LLM errors in a row.

In [ ]:
#| export
# ---------------------------------------------------------------------------- the run
def model_prices(config: dict[str, Any]) -> dict[str, tuple[float, float]]:
    return {m: (float(p[0]), float(p[1]))
            for m, p in config["evaluation"]["prices_per_million"].items()}  # fmt: skip

In [ ]:
#| export
def cost_usd(model: str, tokens_in: int, tokens_out: int, config: dict[str, Any]) -> float:
    pin, pout = model_prices(config).get(model, (0.0, 0.0))
    return round((tokens_in * pin + tokens_out * pout) / 1e6, 6)

In [ ]:
#| export
def prompts_hash(v: Variant) -> str:
    """Changes whenever a system prompt or the variant changes, so runs stay comparable."""
    from src.agents import consumer, humanizer, merchant, platform

    texts = [getattr(m, n) for m in (consumer, humanizer, merchant, platform)
             for n in dir(m) if n.endswith("_SYSTEM")]  # fmt: skip
    blob = json.dumps([sorted(texts), SINGLE_SYSTEM, v.__dict__], sort_keys=True, default=str)
    return hashlib.sha256(blob.encode()).hexdigest()[:12]

In [ ]:
#| export
def fresh_db() -> Path:
    return Path(tempfile.mkdtemp()) / "memory.sqlite"

In [ ]:
#| export
def items_for(bench: Benchmark, split: str, categories: set[str] | None, scripts: bool):
    """(script id or None, turn) pairs: single queries, then the memory scripts' turns."""

    def keep(q):
        return (split == "all" or q["split"] == split) and (
            not categories or q["category"] in categories
        )

    out = [(None, q) for q in bench.queries if keep(q)]
    if scripts:
        for s in bench.scripts:
            if keep({"split": s["split"], "category": "memory"}):
                out += [(s["id"], t) for t in s["turns"]]
    return out

In [ ]:
#| export
def run(
    config: dict[str, Any],
    variant_ids: list[str],
    split: str = "dev",
    repeats: int = 1,
    label: str = "run",
    categories: set[str] | None = None,
    scripts: bool = True,
    limit: int | None = None,
    model_override: str | None = None,
    max_cost: float | None = None,
    cache: bool = True,
    out_dir: Path | None = None,
    bench: Benchmark | None = None,
    log=print,
) -> Path:
    """Run the variants on the benchmark; returns the results folder."""
    bench = bench or load_benchmark(config)
    all_v = variants(config)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    folder = out_dir or repo_path(config["evaluation"]["results_dir"]) / f"{stamp}_{label}"
    folder.mkdir(parents=True, exist_ok=True)
    columns = RESULTS_CSV_COLUMNS + EXTRA_COLUMNS
    items = items_for(bench, split, categories, scripts)[: limit or None]
    meta = {"label": label, "started": stamp, "split": split, "repeats": repeats,
            "variants": variant_ids, "model_override": model_override, "items": len(items),
            "benchmark": str(bench.data.meta.get("folder")), "cache": cache}  # fmt: skip
    (folder / "run.json").write_text(json.dumps(meta, indent=1))
    spent, failures = 0.0, 0
    with (folder / "results.csv").open("w", newline="") as fcsv, \
         (folder / "replies.jsonl").open("w", encoding="utf-8") as fj:  # fmt: skip
        writer = csv.DictWriter(fcsv, columns, extrasaction="ignore")
        writer.writeheader()
        for vid in variant_ids:
            v = all_v[vid]
            model_key = model_override or v.model
            cfg = variant_config(config, v)
            for r in range(repeats):
                llm = eval_llm(cfg, model_key, r, cache)
                system = (SingleAgent if v.agents == "single" else MultiAgent)(cfg, v, bench.data,
                                                                               llm)  # fmt: skip
                memories: dict[str, BuyerMemory] = {}
                for script, q in items:
                    if max_cost is not None and spent >= max_cost:
                        log(f"stopped: ${spent:.2f} spent, the cap is ${max_cost:.2f}")
                        meta["stopped_at_cost"] = round(spent, 4)
                        (folder / "run.json").write_text(json.dumps(meta, indent=1))
                        return folder
                    before = llm.usage()
                    t0 = time.perf_counter()
                    out = Outcome()
                    try:
                        if isinstance(system, SingleAgent):
                            out = system.ask(q["text"], q.get("clarify_answer"))
                        else:
                            # a fresh memory per query; one per script when memory is on
                            if script is not None and v.memory:
                                if script not in memories:
                                    memories[script] = BuyerMemory(fresh_db())
                                mem = memories[script]
                            else:
                                mem = BuyerMemory(fresh_db())
                            sid = f"{vid}-{script or q['id']}-{r}"
                            out = system.ask(system.session(mem, sid), q["text"],
                                             q.get("clarify_answer"))  # fmt: skip
                        failures = 0
                    except LLMError as e:
                        out.error = str(e)[:300]
                        failures += 1
                    except Exception as e:  # noqa: BLE001 - a crash is a result too: recorded
                        out.error = f"{type(e).__name__}: {e}"[:300]
                    latency = time.perf_counter() - t0
                    after = llm.usage()
                    t_in = after["prompt_tokens"] - before["prompt_tokens"]
                    t_out = after["completion_tokens"] - before["completion_tokens"]
                    cost = cost_usd(llm.model, t_in, t_out, config)
                    spent += cost
                    row = {"run_id": folder.name, "query_id": q["id"], "config_id": vid,
                           "repeat": r, "model_id": llm.model, "latency_sec": round(latency, 3),
                           "prompt_tokens": t_in, "completion_tokens": t_out, "cost_usd": cost,
                           "prompt_hash": prompts_hash(v), "split": q["split"],
                           "category": q["category"], "expect": q["key"]["expect"],
                           "llm_calls": after["calls"] - before["calls"],
                           "cached_calls": after["cached_calls"] - before["cached_calls"],
                           "error": out.error}  # fmt: skip
                    if out.error is None:
                        row.update(score(q, out, bench.data, config))
                        if script and q["id"].endswith("b"):
                            row["memory_ok"] = int(row["top1_accurate"] and not out.clarify_asked)
                    writer.writerow(row)
                    fj.write(
                        json.dumps(
                            {
                                "query_id": q["id"],
                                "config_id": vid,
                                "repeat": r,
                                "text": q["text"],
                                "reply": out.reply,
                                "picks": out.picks,
                                "state": out.state,
                                "error": out.error,
                            },
                            ensure_ascii=False,
                        )
                        + "\n"
                    )
                    fcsv.flush()
                    fj.flush()
                    log(f"{vid:16} r{r} {q['id']:5} top1={row.get('top1_accurate', '-')} "
                        f"{latency:5.1f}s ${spent:.4f}" + (f" ERROR {out.error[:80]}"
                                                           if out.error else ""))  # fmt: skip
                    if failures >= 3:
                        log("stopped: 3 LLM errors in a row (quota or network)")
                        meta["stopped_on_errors"] = out.error
                        (folder / "run.json").write_text(json.dumps(meta, indent=1))
                        return folder
    meta["cost_usd"] = round(spent, 4)
    meta["finished"] = datetime.now().strftime("%Y%m%d_%H%M%S")
    (folder / "run.json").write_text(json.dumps(meta, indent=1))
    return folder

In [ ]:
#| export
def read_rows(folder: Path) -> list[dict[str, Any]]:
    with (folder / "results.csv").open(newline="") as f:
        return list(csv.DictReader(f))

## The judge

The judge sees the request and the reply only (not the variant). Before its scores count, two people rate 20 replies (`rating-sheet`) and `agreement` compares them.

In [ ]:
#| export
# ---------------------------------------------------------------------------- the judge
class JudgeScore(BaseModel):
    naturalness: int = Field(ge=1, le=5)  # sounds like a helpful person talking
    helpfulness: int = Field(ge=1, le=5)  # answers what was asked, clearly
    reason: str = ""

In [ ]:
#| export
JUDGE_SYSTEM = """You rate one spoken reply from a Pokemon card shopping assistant.
naturalness (1-5): 5 sounds like a friendly, knowledgeable person talking (short, plain,
warm); 1 reads like a report, a list or a robot.
helpfulness (1-5): 5 answers the buyer's request clearly (which card, price, why, any
warning); 1 doesn't answer it.
Judge only the wording, not whether the listings are good. Give one short reason."""

In [ ]:
#| export
@offline_handler("judge")
def offline_judge(_system: str, prompt: str) -> dict:
    """Offline stand-in: longer replies sound less natural."""
    words = len(between(prompt, "reply").split())
    return {"naturalness": 5 if words <= 60 else 4 if words <= 90 else 3,
            "helpfulness": 4, "reason": "offline rule"}  # fmt: skip

In [ ]:
#| export
def judge(folder: Path, config: dict[str, Any], llm: LLM | None = None, log=print) -> Path:
    """Rate every reply with `models.judge` (a different family from the agents' Gemini);
    writes judge.jsonl and fills naturalness_score in results.csv."""
    llm = llm or eval_llm(config, config["evaluation"]["judge_model"], 0)
    disclosure = config["response"]["ai_disclosure"]  # the graph's first-reply AI notice
    scores: dict[tuple, dict] = {}
    with (folder / "replies.jsonl").open(encoding="utf-8") as f, \
         (folder / "judge.jsonl").open("w", encoding="utf-8") as out:  # fmt: skip
        for line in f:
            rec = json.loads(line)
            if not rec.get("reply"):
                continue
            reply = rec["reply"].removeprefix(disclosure).strip()  # same for every variant
            prompt = f"<request>{rec['text']}</request>\n<reply>{reply}</reply>"
            try:
                s = llm.json(tag="judge", system=JUDGE_SYSTEM, prompt=prompt, schema=JudgeScore)
            except LLMError as e:
                log(f"judge stopped: {e}")
                break
            key = (rec["query_id"], rec["config_id"], str(rec["repeat"]))
            scores[key] = s.model_dump()
            out.write(json.dumps({**rec, **scores[key]}, ensure_ascii=False) + "\n")
    rows = read_rows(folder)
    for row in rows:
        s = scores.get((row["query_id"], row["config_id"], row["repeat"]))
        if s:
            row["naturalness_score"] = s["naturalness"]
    with (folder / "results.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, RESULTS_CSV_COLUMNS + EXTRA_COLUMNS, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)
    usage = llm.usage()
    log(
        f"judged {len(scores)} replies, ${cost_usd(llm.model, usage['prompt_tokens'], usage['completion_tokens'], config):.4f}"
    )
    return folder / "judge.jsonl"

In [ ]:
#| export
def rating_sheet(folder: Path, out: Path, n: int = 20, seed: int = 7) -> Path:
    """`n` replies (spread over the variants) for two people to rate 1-5, blind to the
    variant: the check that the judge agrees with humans before its scores are used."""
    recs = [json.loads(x) for x in (folder / "judge.jsonl").read_text().splitlines()]
    random.Random(seed).shuffle(recs)
    seen: dict[str, int] = defaultdict(int)
    picked = []
    per_variant = max(1, n // max(1, len({r["config_id"] for r in recs})))
    for r in recs:
        if seen[r["config_id"]] < per_variant and len(picked) < n:
            seen[r["config_id"]] += 1
            picked.append(r)
    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["item", "query_id", "config_id", "repeat", "request", "reply",
                    "rater1_naturalness", "rater2_naturalness"])  # fmt: skip
        for i, r in enumerate(picked, 1):
            w.writerow([i, r["query_id"], r["config_id"], r["repeat"], r["text"], r["reply"], "",
                        ""])  # fmt: skip
    return out

In [ ]:
#| export
def ranks(xs: list[float]) -> list[float]:
    order = sorted(range(len(xs)), key=lambda i: xs[i])
    r = [0.0] * len(xs)
    i = 0
    while i < len(xs):
        j = i
        while j + 1 < len(xs) and xs[order[j + 1]] == xs[order[i]]:
            j += 1
        for k in range(i, j + 1):
            r[order[k]] = (i + j) / 2 + 1
        i = j + 1
    return r

In [ ]:
#| export
def spearman(a: list[float], b: list[float]) -> float | None:
    if len(a) < 3:
        return None
    ra, rb = ranks(a), ranks(b)
    ma, mb = statistics.fmean(ra), statistics.fmean(rb)
    num = sum((x - ma) * (y - mb) for x, y in zip(ra, rb, strict=True))
    den = math.sqrt(sum((x - ma) ** 2 for x in ra) * sum((y - mb) ** 2 for y in rb))
    return round(num / den, 3) if den else None

In [ ]:
#| export
def judge_agreement(folder: Path, sheet: Path) -> dict[str, Any]:
    """How well the judge matches the people who filled in the rating sheet."""
    judged = {(r["query_id"], r["config_id"], str(r["repeat"])): r["naturalness"]
              for r in map(json.loads, (folder / "judge.jsonl").read_text().splitlines())}  # fmt: skip
    pairs = []
    with sheet.open(newline="") as f:
        for row in csv.DictReader(f):
            people = [float(row[c]) for c in ("rater1_naturalness", "rater2_naturalness")
                      if row.get(c, "").strip()]  # fmt: skip
            j = judged.get((row["query_id"], row["config_id"], row["repeat"]))
            if people and j is not None:
                pairs.append((statistics.fmean(people), float(j)))
    if not pairs:
        return {"rated": 0}
    human, model = [p[0] for p in pairs], [p[1] for p in pairs]
    return {"rated": len(pairs), "spearman": spearman(human, model),
            "within_1_point": round(sum(abs(a - b) <= 1 for a, b in pairs) / len(pairs), 3),
            "mean_human": round(statistics.fmean(human), 2),
            "mean_judge": round(statistics.fmean(model), 2)}  # fmt: skip

## Statistics

A query is the unit: each metric is averaged over repeats per query, then over queries, with a bootstrap 95% interval. Each variant is compared with `full` on the same queries.

In [ ]:
#| export
# ---------------------------------------------------------------------------- statistics
def num(row: dict, col: str) -> float | None:
    v = row.get(col, "")
    return float(v) if v not in ("", None) else None

In [ ]:
#| export
def per_query(rows: list[dict], col: str) -> dict[str, float]:
    """The metric per query, averaged over repeats (the unit for confidence intervals)."""
    acc: dict[str, list[float]] = defaultdict(list)
    for r in rows:
        v = num(r, col)
        if v is not None:
            acc[r["query_id"]].append(v)
    return {q: statistics.fmean(v) for q, v in acc.items()}

In [ ]:
#| export
def bootstrap_ci(values: list[float], n: int = 2000, seed: int = 0) -> tuple[float, float]:
    if not values:
        return (math.nan, math.nan)
    rng = random.Random(seed)
    means = sorted(statistics.fmean(rng.choices(values, k=len(values))) for _ in range(n))
    return (round(means[int(0.025 * n)], 4), round(means[int(0.975 * n) - 1], 4))

In [ ]:
#| export
def mcnemar(a: dict[str, float], b: dict[str, float]) -> dict[str, Any]:
    """Exact McNemar test on paired per-query correctness (a query counts as correct when it
    was right in most repeats)."""
    common = sorted(set(a) & set(b))
    only_a = sum(a[q] > 0.5 >= b[q] for q in common)
    only_b = sum(b[q] > 0.5 >= a[q] for q in common)
    n = only_a + only_b
    p = (
        min(1.0, 2 * sum(math.comb(n, k) for k in range(min(only_a, only_b) + 1)) / 2**n)
        if n
        else 1.0
    )
    return {"queries": len(common), "only_baseline_right": only_a, "only_variant_right": only_b,
            "p_value": round(p, 4)}  # fmt: skip

In [ ]:
#| export
METRICS = ["top1_accurate", "top3_accurate", "correct_refusal", "policy_violation",
           "trap_recommended", "arithmetic_error", "scam_recall", "scam_fpr", "memory_ok",
           "naturalness_score", "clarify_asked"]  # fmt: skip

In [ ]:
#| export
def summarize(folder: Path, baseline: str = "full") -> dict[str, Path]:
    """summary.csv (each variant: every metric with a 95% bootstrap CI over queries, latency,
    tokens, cost), by_category.csv, comparisons.csv (each variant against `baseline`: change
    in top-1 accuracy with a paired bootstrap CI, McNemar p) and report.md."""
    rows = [r for r in read_rows(folder) if not r.get("error")]
    by_v: dict[str, list[dict]] = defaultdict(list)
    for r in rows:
        by_v[r["config_id"]].append(r)
    summary, cats, comps = [], [], []
    for vid, vr in by_v.items():
        s: dict[str, Any] = {"variant": vid, "model": vr[0]["model_id"],
                             "queries": len({r["query_id"] for r in vr}), "rows": len(vr)}  # fmt: skip
        for m in METRICS:
            pq = per_query(vr, m)
            if pq:
                s[m] = round(statistics.fmean(pq.values()), 4)
                lo, hi = bootstrap_ci(list(pq.values()))
                s[f"{m}_ci"] = f"{lo:.3f}-{hi:.3f}"
        lat = sorted(float(r["latency_sec"]) for r in vr if r["cached_calls"] in ("0", 0))
        if lat:
            s["latency_median_s"] = round(statistics.median(lat), 2)
            s["latency_p90_s"] = round(lat[int(0.9 * (len(lat) - 1))], 2)
        s["tokens_per_query"] = round(statistics.fmean(
            int(r["prompt_tokens"]) + int(r["completion_tokens"]) for r in vr))  # fmt: skip
        s["cost_per_query_usd"] = round(statistics.fmean(float(r["cost_usd"]) for r in vr), 5)
        s["cost_usd"] = round(sum(float(r["cost_usd"]) for r in vr), 4)
        summary.append(s)
        for cat in sorted({r["category"] for r in vr}):
            cr = [r for r in vr if r["category"] == cat]
            cats.append({"variant": vid, "category": cat, "queries": len({r["query_id"] for r in cr}),
                         **{m: round(statistics.fmean(per_query(cr, m).values()), 4)
                            for m in ("top1_accurate", "top3_accurate", "policy_violation")
                            if per_query(cr, m)}})  # fmt: skip
    base = per_query(by_v.get(baseline, []), "top1_accurate")
    for vid, vr in by_v.items():
        if vid == baseline or not base:
            continue
        other = per_query(vr, "top1_accurate")
        common = sorted(set(base) & set(other))
        diffs = [other[q] - base[q] for q in common]
        lo, hi = bootstrap_ci(diffs)
        comps.append({"variant": vid, "baseline": baseline,
                      "top1_change": round(statistics.fmean(diffs), 4) if diffs else None,
                      "top1_change_ci": f"{lo:.3f}-{hi:.3f}", **mcnemar(base, other)})  # fmt: skip
    files = {"summary": folder / "summary.csv", "by_category": folder / "by_category.csv",
             "comparisons": folder / "comparisons.csv"}  # fmt: skip
    for name, data in (("summary", summary), ("by_category", cats), ("comparisons", comps)):
        cols = list(dict.fromkeys(k for d in data for k in d))
        with files[name].open("w", newline="") as f:
            w = csv.DictWriter(f, cols)
            w.writeheader()
            w.writerows(data)
    files["report"] = write_report(folder, summary, cats, comps)
    return files

In [ ]:
#| export
def write_report(folder: Path, summary: list[dict], cats: list[dict], comps: list[dict]) -> Path:
    meta = json.loads((folder / "run.json").read_text())

    def pct(v):
        return "–" if v is None else f"{100 * v:.0f}%"

    lines = [f"# Evaluation run {folder.name}", "",
             (f"Split: {meta.get('split')} · repeats: {meta.get('repeats')} · "
             f"model override: {meta.get('model_override') or 'none'}"), "",
             ("| Variant | Model | Top-1 (95% CI) | Top-3 | Policy violations | Traps recommended | "
             "Scam recall | Decoy FPR | Arithmetic errors | Naturalness | Median latency | "
             "Cost/query |"), "|" + "---|" * 12]  # fmt: skip
    for s in summary:
        lines.append(
            f"| {s['variant']} | {s['model']} | {pct(s.get('top1_accurate'))} "
            f"({s.get('top1_accurate_ci', '')}) | {pct(s.get('top3_accurate'))} | "
            f"{pct(s.get('policy_violation'))} | {pct(s.get('trap_recommended'))} | "
            f"{pct(s.get('scam_recall'))} | {pct(s.get('scam_fpr'))} | "
            f"{pct(s.get('arithmetic_error'))} | {s.get('naturalness_score', '–')} | "
            f"{s.get('latency_median_s', '–')} s | ${s.get('cost_per_query_usd', 0):.4f} |"
        )
    if comps:
        lines += ["", "## Against the full system (paired, same queries)", "",
                  ("| Variant | Top-1 change (95% CI) | Only full right | Only variant right | "
                  "McNemar p |"), "|---|---|---|---|---|"]  # fmt: skip
        lines += [f"| {c['variant']} | {c['top1_change']:+.3f} ({c['top1_change_ci']}) | "
                  f"{c['only_baseline_right']} | {c['only_variant_right']} | {c['p_value']} |"
                  for c in comps if c["top1_change"] is not None]  # fmt: skip
    lines += ["", "## Top-1 accuracy by query type", "", "| Variant | Category | Queries | Top-1 |",
              "|---|---|---|---|"]  # fmt: skip
    lines += [f"| {c['variant']} | {c['category']} | {c['queries']} | "
              f"{pct(c.get('top1_accurate'))} |" for c in cats]  # fmt: skip
    path = folder / "report.md"
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return path

In [ ]:
#| export
def screening(config: dict[str, Any], bench: Benchmark | None = None) -> dict[str, Any]:
    """The Merchant's scam check on every benchmark listing, by split: how many traps it
    rejects or warns about (recall), and how many honest decoys and clean listings it rejects
    (false positives). Deterministic: no LLM."""
    bench = bench or load_benchmark(config)
    out: dict[str, Any] = {}
    for split in ("dev", "test", "all"):
        xs = [x for x in bench.data.listings if split == "all" or x.split == split]
        dec = {x.listing_id: assess(x, bench.data.prices, bench.data.red_flags, config).decision
               for x in xs}  # fmt: skip

        def rate(label, decisions, xs=xs, dec=dec):
            group = [x for x in xs if x.label == label]
            return round(sum(dec[x.listing_id] in decisions for x in group) / len(group), 3)

        out[split] = {"trap_recall": rate("trap", {"reject", "warn"}),
                      "trap_rejected": rate("trap", {"reject"}),
                      "decoy_rejected": rate("decoy", {"reject"}),
                      "decoy_warned": rate("decoy", {"warn"}),
                      "clean_rejected": rate("clean", {"reject"}),
                      "missed_traps": sorted(x.listing_id for x in xs if x.label == "trap"
                                             and dec[x.listing_id] == "ok")}  # fmt: skip
    return out

In [ ]:
#| export
# ---------------------------------------------------------------------------- the estimate
def single_agent_tokens(config: dict[str, Any], reply_tokens: int = 250) -> tuple[int, int]:
    """The single agent's tokens per query without running it: its prompt (every listing with
    its tool outputs) at about 4 characters per token, and a short reply."""
    bench = load_benchmark(config)
    table = json.dumps([listing_tools(x, bench.data, config) for x in bench.data.listings],
                       ensure_ascii=False)  # fmt: skip
    return (len(SINGLE_SYSTEM) + len(table) + 200) // 4, reply_tokens

## The cost estimate

Tokens per query from a pilot, by architecture, priced at each variant's own model.

In [ ]:
#| export
def estimate(
    config: dict[str, Any], pilot: Path, variant_ids: list[str], queries: int, repeats: int
) -> list[dict[str, Any]]:
    """What a full run would cost: the pilot's tokens per query for each architecture, at
    each variant's own model price. A rough guide: Gemini spends some extra output tokens on
    thinking, which a Groq pilot doesn't show."""
    rows = [r for r in read_rows(pilot) if not r.get("error") and int(r["prompt_tokens"] or 0)]
    per_arch: dict[str, list[tuple[int, int]]] = defaultdict(list)
    all_v = variants(config)
    for r in rows:
        arch = all_v[r["config_id"]].agents if r["config_id"] in all_v else "multi"
        per_arch[arch].append((int(r["prompt_tokens"]), int(r["completion_tokens"])))
    if not per_arch.get("single"):  # not in the pilot: its prompt size is known anyway
        per_arch["single"] = [single_agent_tokens(config)]
    out = []
    for vid in variant_ids:
        v = all_v[vid]
        sample = per_arch.get(v.agents) or per_arch.get("multi") or [(0, 0)]
        t_in = statistics.fmean(a for a, _ in sample)
        t_out = statistics.fmean(b for _, b in sample)
        model = config["models"][v.model]
        n = queries * repeats
        out.append({"variant": vid, "model": model, "query_runs": n,
                    "tokens_in_per_query": round(t_in), "tokens_out_per_query": round(t_out),
                    "cost_usd": round(n * cost_usd(model, int(t_in), int(t_out), config), 2)})  # fmt: skip
    return out

## Command line

In [ ]:
#| export
# ---------------------------------------------------------------------------- command line
def main(argv: list[str] | None = None) -> int:
    """`python -m src.evaluation <command>`: build, run, judge, stats, estimate, screen,
    rating-sheet, agreement. In VS Code: Run and Debug → "Evaluation: dev run (offline)"."""
    ap = argparse.ArgumentParser(
        prog="python -m src.evaluation", description=__doc__.split("\n")[0]
    )
    sub = ap.add_subparsers(dest="cmd", required=True)
    sub.add_parser("build", help="write the benchmark files to evaluation.benchmark_dir")
    r = sub.add_parser("run", help="run variants on the benchmark")
    r.add_argument("--variants", default="full", help="comma-separated ids, or all")
    r.add_argument("--split", default="dev", choices=["dev", "test", "all"])
    r.add_argument("--repeats", type=int, default=1)
    r.add_argument("--label", default="run")
    r.add_argument("--categories", help="comma-separated: simple,complex,consultative,memory")
    r.add_argument("--no-scripts", action="store_true", help="skip the memory scripts")
    r.add_argument("--limit", type=int, help="only the first N queries (a quick check)")
    r.add_argument("--model", help="run every variant on this model key (e.g. small, offline)")
    r.add_argument("--max-cost", type=float, help="stop once this many dollars are spent")
    r.add_argument("--no-cache", action="store_true", help="fresh LLM calls even if cached")
    j = sub.add_parser("judge", help="rate the replies of a run with the judge model")
    j.add_argument("folder")
    j.add_argument("--model", help="judge model key (default evaluation.judge_model)")
    s = sub.add_parser("stats", help="summary, comparisons and report.md for a run")
    s.add_argument("folder")
    s.add_argument("--baseline", default="full")
    e = sub.add_parser("estimate", help="cost of a full run, from a pilot run's tokens")
    e.add_argument("folder")
    e.add_argument("--variants", default="all")
    e.add_argument("--queries", type=int, default=60, help="queries + memory-script turns")
    e.add_argument("--repeats", type=int, default=3)
    sub.add_parser("screen", help="the scam check on every benchmark listing")
    rs = sub.add_parser("rating-sheet", help="20 judged replies for two people to rate")
    rs.add_argument("folder")
    ag = sub.add_parser("agreement", help="judge vs the filled-in rating sheet")
    ag.add_argument("folder")
    ag.add_argument("sheet")
    args = ap.parse_args(argv)

    cfg = load_config()
    all_ids = list(cfg["evaluation"]["variants"])

    def ids(text):
        return all_ids if text == "all" else [x.strip() for x in text.split(",") if x.strip()]

    if args.cmd == "build":
        print(f"benchmark written to {build_benchmark(cfg)}")
    elif args.cmd == "run":
        folder = run(cfg, ids(args.variants), args.split, args.repeats, args.label,
                     set(args.categories.split(",")) if args.categories else None,
                     not args.no_scripts, args.limit, args.model, args.max_cost,
                     not args.no_cache)  # fmt: skip
        files = summarize(folder)
        print(f"\nresults: {folder}\n{files['report'].read_text()}")
    elif args.cmd == "judge":
        llm = eval_llm(cfg, args.model, 0) if args.model else None
        judge(Path(args.folder), cfg, llm)
        summarize(Path(args.folder))
    elif args.cmd == "stats":
        files = summarize(Path(args.folder), args.baseline)
        print(files["report"].read_text())
    elif args.cmd == "estimate":
        rows = estimate(cfg, Path(args.folder), ids(args.variants), args.queries, args.repeats)
        for row in rows:
            print(
                f"{row['variant']:18} {row['model']:26} {row['query_runs']:4} runs  "
                f"~{row['tokens_in_per_query']:6} in + {row['tokens_out_per_query']:5} out "
                f"tokens/query  ${row['cost_usd']:.2f}"
            )
        print(f"total ≈ ${sum(r['cost_usd'] for r in rows):.2f}")
    elif args.cmd == "screen":
        print(json.dumps(screening(cfg), indent=1))
    elif args.cmd == "rating-sheet":
        out = rating_sheet(Path(args.folder), Path(args.folder) / "human_ratings.csv")
        print(f"fill in {out} (two raters, 1-5), then run: agreement {args.folder} {out}")
    elif args.cmd == "agreement":
        print(json.dumps(judge_agreement(Path(args.folder), Path(args.sheet)), indent=1))
    return 0

In [ ]:
#| export
if __name__ == "__main__" and "ipykernel" not in sys.modules:  # not inside a notebook
    sys.exit(main())

# The scraping pipeline (`src/eval_pipeline.py`)

The scrape, the scope decisions and the card identification against hand labels. `eval/data/sample_raw_labels.csv` labels the 28 synthetic raw listings; the hand check of the 100-listing snapshot gives the real labels (`sheet` makes the pre-filled sheet).

In [ ]:
#| export eval_pipeline
"""How good the data pipeline is (Section 10): the scrape itself, the scope decisions and the
card identification, checked against hand labels.

- `scrape_health()`: what a raw Apify scrape gives us, without any labels: how many items are
  real listings (not sold, hidden or $1 placeholders), duplicates, and how often each field we
  rely on is there (description, condition, photos, location, listing date, delivery).
- `label_sheet()`: a CSV for the hand check, one row per real listing, pre-filled with what the
  pipeline decided, so the person only corrects it. Columns: in_scope (1/0), out_reason, then
  the card: card_name, set_code (TCGdex id), number, variant, condition.
- `score_pipeline()`: runs the pipeline (pre-filter, LLM identification, scope filter, and with
  a PriceBook the set check and prices) on the scrape and compares it with the labels: scope
  precision and recall, each identity field's accuracy on the listings both kept, the share
  fully right (name, set and number), price coverage, LLM calls, time and cost per listing.

`eval/data/sample_raw_labels.csv` labels the 28 synthetic raw listings, so this runs now; the
100-listing snapshot gets its own labels in the hand check.
"""

import argparse
import csv
import json
import sys
import time
from collections import Counter
from pathlib import Path
from typing import Any

from rapidfuzz import fuzz

from src.config import load_config
from src.llm import LLM, make_llm
from src.pipeline import PriceBook, anon_id, anonymize, first, identify_listings, to_price
from src.schemas import CardListing

In [ ]:
#| export eval_pipeline
LABEL_COLUMNS = ["source_id", "in_scope", "out_reason", "card_name", "set_code", "number",
                 "variant", "condition", "note", "title", "description"]  # fmt: skip
FIELDS = ["card_name", "set_code", "number", "variant", "condition"]

In [ ]:
#| export eval_pipeline
def scrape_health(items: list[dict], config: dict[str, Any]) -> dict[str, Any]:
    """What a raw scrape contains, before any LLM."""
    ids = [first(i, "id", "listing_id") for i in items]
    usable = [r for r in (anonymize(i, config) for i in items) if r]
    floor = float(config["ingest"]["placeholder_max_price"])

    def share(n):
        return round(n / len(items), 3) if items else 0.0

    prices = [to_price(first(i, "listingPrice", "listing_price", "price")) for i in items]
    out = {
        "items": len(items),
        "usable": len(usable),
        "usable_share": share(len(usable)),
        "sold": sum(first(i, "isSold", "is_sold") is True for i in items),
        "hidden": sum(first(i, "isHidden") is True for i in items),
        "placeholder_price": sum(p is not None and 0 < p <= floor for p in prices),
        "duplicates": len(ids) - len(set(ids)),
        "has_field": {
            "description": share(sum(bool(r["raw_description"]) for r in usable)),
            "condition": share(sum(bool(first(i, "condition")) for i in items)),
            "photo": share(sum((r.get("photo_count") or 0) > 0 for r in usable)),
            "location": share(sum(bool(r["location_city"]) for r in usable)),
            "listed_date": share(sum(bool(r.get("listed_at")) for r in usable)),
            "distance": share(sum(r.get("distance_miles") is not None for r in usable)),
            "delivery": share(sum(r["delivery_type"] != "unknown" for r in usable)),
        },
    }
    out["note"] = "has_field shares are over all items, of the usable listings' fields"
    return out

In [ ]:
#| export eval_pipeline
def load_labels(path: Path) -> dict[str, dict[str, str]]:
    with path.open(newline="", encoding="utf-8") as f:
        return {row["source_id"]: row for row in csv.DictReader(f)}

In [ ]:
#| export eval_pipeline
def label_sheet(
    items: list[dict], out: Path, config: dict[str, Any], kept: list[CardListing] | None = None
) -> Path:
    """The hand-check sheet: one row per real listing, pre-filled from `kept` (what the
    pipeline identified); a listing it dropped starts as out of scope."""
    by_anon = {x.listing_id: x for x in (kept or [])}
    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, LABEL_COLUMNS)
        w.writeheader()
        for item in items:
            row = anonymize(item, config)
            if not row:
                continue
            sid = str(first(item, "id", "listing_id"))
            x = by_anon.get(anon_id(sid))
            w.writerow({
                "source_id": sid, "in_scope": int(x is not None),
                "out_reason": "" if x else "dropped by the pipeline: check",
                **({f: getattr(x, f) for f in FIELDS} if x else {}),
                "title": row["raw_title"], "description": row["raw_description"][:300],
            })  # fmt: skip
    return out

In [ ]:
#| export eval_pipeline
def same(field: str, truth: str, guess: Any) -> bool:
    guess = "" if guess is None else str(guess)
    if field == "card_name":
        return fuzz.token_set_ratio(truth.casefold(), guess.casefold()) >= 90 and (
            truth.casefold().split()[-1] == guess.casefold().split()[-1] if guess else False
        )
    if field == "number":
        return truth.lstrip("0") == guess.lstrip("0")
    return truth.casefold() == guess.casefold()

In [ ]:
#| export eval_pipeline
def score_pipeline(
    config: dict[str, Any],
    items: list[dict],
    labels: dict[str, dict[str, str]],
    llm: LLM,
    pricebook: PriceBook | None = None,
) -> dict[str, Any]:
    """Run the pipeline on the scrape and compare it with the labels."""
    rows = [r for r in (anonymize(i, config) for i in items) if r]
    t0 = time.perf_counter()
    before = llm.usage()
    kept, dropped, stats = identify_listings(config, llm, rows, log=lambda *_: None)
    if pricebook is not None:
        pricebook.resolve_sets(kept)
        pricebook.ensure_prices(kept, pause_s=0.05)
        pricebook.verify_cards(kept)
    seconds = time.perf_counter() - t0
    after = llm.usage()
    by_anon = {x.listing_id: x for x in kept}
    per_listing, tp, fp, fn = [], 0, 0, 0
    field_hits: Counter = Counter()
    full_hits = priced = 0
    for item in items:
        sid = str(first(item, "id", "listing_id"))
        lab = labels.get(sid)
        if lab is None or anonymize(item, config) is None:
            continue
        x = by_anon.get(anon_id(sid))
        in_scope = lab["in_scope"].strip() == "1"
        rec: dict[str, Any] = {
            "source_id": sid,
            "in_scope": int(in_scope),
            "kept": int(x is not None),
        }
        if in_scope and x is not None:
            tp += 1
            for fld in FIELDS:
                ok = same(fld, lab[fld], getattr(x, fld))
                field_hits[fld] += ok
                rec[fld] = f"{getattr(x, fld)}" + ("" if ok else f" (truth {lab[fld]})")
            full_hits += all(
                same(f, lab[f], getattr(x, f)) for f in ("card_name", "set_code", "number")
            )
            has_price = bool(pricebook and pricebook.prices.get(x.card_key or ""))
            priced += has_price
            rec["priced"] = int(has_price)
        elif in_scope:
            fn += 1
        elif x is not None:
            fp += 1
        per_listing.append(rec)
    n_in = tp + fn
    calls = after["calls"] - before["calls"]
    tokens = (after["prompt_tokens"] - before["prompt_tokens"],
              after["completion_tokens"] - before["completion_tokens"])  # fmt: skip
    from src.evaluation import cost_usd

    result = {
        "usable": len(rows), "labeled": len(per_listing), "in_scope": n_in,
        "kept_in_scope": tp, "kept_out_of_scope": fp, "dropped_in_scope": fn,
        "scope_precision": round(tp / (tp + fp), 3) if tp + fp else None,
        "scope_recall": round(tp / n_in, 3) if n_in else None,
        "field_accuracy": {f: round(field_hits[f] / tp, 3) if tp else None for f in FIELDS},
        "fully_right": round(full_hits / tp, 3) if tp else None,
        "price_coverage": round(priced / tp, 3) if tp and pricebook else None,
        "dropped": dropped, "identify": stats, "llm_calls": calls, "model": llm.model,
        "seconds": round(seconds, 2),
        "seconds_per_listing": round(seconds / len(rows), 3) if rows else None,
        "cost_usd": cost_usd(llm.model, *tokens, config),
    }  # fmt: skip
    result["per_listing"] = per_listing
    return result

In [ ]:
#| export eval_pipeline
def main(argv: list[str] | None = None) -> int:
    """`python -m src.eval_pipeline health|sheet|score RAW ...`"""
    ap = argparse.ArgumentParser(
        prog="python -m src.eval_pipeline", description=__doc__.split("\n")[0]
    )
    sub = ap.add_subparsers(dest="cmd", required=True)
    h = sub.add_parser("health", help="what a raw scrape contains")
    h.add_argument("raw", nargs="+")
    s = sub.add_parser("sheet", help="the hand-check sheet for a scrape")
    s.add_argument("raw")
    s.add_argument("--out", required=True)
    s.add_argument("--model", help="identify with this model key first (pre-fills the sheet)")
    c = sub.add_parser("score", help="the pipeline against hand labels")
    c.add_argument("raw")
    c.add_argument("labels")
    c.add_argument("--model", help="model key for identification (default ingest.llm / llm.active)")
    c.add_argument("--prices", action="store_true", help="also check sets and prices on TCGdex")
    c.add_argument("--out", help="write the result (with each listing) to this JSON file")
    args = ap.parse_args(argv)
    cfg = load_config()

    def items_of(path):
        return json.loads(Path(path).read_text(encoding="utf-8"))

    if args.cmd == "health":
        for path in args.raw:
            print(path, json.dumps(scrape_health(items_of(path), cfg), indent=1))
    elif args.cmd == "sheet":
        items = items_of(args.raw)
        kept = None
        if args.model:
            rows = [r for r in (anonymize(i, cfg) for i in items) if r]
            kept, _, _ = identify_listings(cfg, make_llm(cfg, args.model), rows)
        print(f"fill in {label_sheet(items, Path(args.out), cfg, kept)}")
    elif args.cmd == "score":
        cfg_llm = args.model or cfg["ingest"].get("llm")
        llm = make_llm(cfg, cfg_llm)
        book = PriceBook(cfg, {}) if args.prices else None
        result = score_pipeline(cfg, items_of(args.raw), load_labels(Path(args.labels)), llm, book)
        if args.out:
            Path(args.out).write_text(json.dumps(result, indent=1))
        print(json.dumps({k: v for k, v in result.items() if k != "per_listing"}, indent=1))
        wrong = [r for r in result["per_listing"] if any("truth" in str(v) for v in r.values())]
        if wrong:
            print("wrong fields:", *(json.dumps(r) for r in wrong), sep="\n  ")
    return 0

In [ ]:
#| export eval_pipeline
if __name__ == "__main__" and "ipykernel" not in sys.modules:  # not inside a notebook
    sys.exit(main())

# The voice (`src/eval_voice.py`)

Speech to text on spoken benchmark requests (4 Edge-TTS voices, with and without the vocabulary prompt), spoken vs typed top picks, and H5 from the voice chat's timing log.

In [ ]:
#| export eval_voice
"""How well the voice works (Section 10, H5): speech to text, spoken against typed requests, and
latency.

- `stt` (`run_stt()`): each benchmark request is spoken with Edge-TTS in several voices and
  accents (`evaluation.voices`), then transcribed by the voice chat's `Transcriber` twice:
  with the card-name vocabulary prompt and without it (an ablation). Measured: word error rate,
  entity recall (the card name, set and amounts that were said, found in the transcript) and
  transcription time. With `--agents MODEL`, each transcript also goes through the four agents,
  and its top pick is scored like the typed request's: the voice-to-voice accuracy.
- `timing` (`timing_summary()`): the H5 numbers from the voice chat's own timing log
  (eval/voice_runs.jsonl): time to the transcript, to the first filler and to the first answer
  audio, measured from the moment the buyer stops talking.
- `tts` (`tts_latency()`): how long Edge-TTS takes for the first sentence of real replies.
"""

import argparse
import copy
import csv
import hashlib
import json
import re
import statistics
import sys
import time
from datetime import datetime
from pathlib import Path
from typing import Any

from rapidfuzz import fuzz

from src.agents.merchant import name_matches
from src.benchmark import PRINTINGS, Benchmark, load_benchmark
from src.config import load_config, repo_path

In [ ]:
#| export eval_voice
def normalize(text: str) -> list[str]:
    """Words for the error rate: lower case, no punctuation, "$1,300" -> "1300",
    "base-set" -> "base set", "dollars" dropped (Whisper writes "$500", people say it)."""
    t = text.lower().replace("-", " ").replace("$", " ")
    t = re.sub(r"(?<=\d),(?=\d{3})", "", t)
    t = re.sub(r"[^a-z0-9./ ]+", " ", t)
    t = re.sub(r"(?<![0-9])\.|\.(?![0-9])", " ", t)
    return [w for w in t.split() if w not in {"dollars", "dollar", "bucks"}]

In [ ]:
#| export eval_voice
def wer(reference: str, hypothesis: str) -> float:
    """Word error rate: word-level edit distance over the reference's length."""
    ref, hyp = normalize(reference), normalize(hypothesis)
    if not ref:
        return 0.0 if not hyp else 1.0
    prev = list(range(len(hyp) + 1))
    for i, r in enumerate(ref, 1):
        cur = [i] + [0] * len(hyp)
        for j, h in enumerate(hyp, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (r != h))
        prev = cur
    return round(prev[-1] / len(ref), 4)

In [ ]:
#| export eval_voice
def entities(query: dict) -> list[tuple[str, str]]:
    """What the request says that matters: the card names, set names and amounts in it."""
    text = " ".join(normalize(query["text"]))
    truth = query["truth"]
    out = []
    for pkey in truth["printings"]:
        p = PRINTINGS[pkey]
        if name_matches(p["card_name"].split()[0], text, 90):
            out.append(("card", p["card_name"].split()[0]))
        if fuzz.partial_ratio(p["set_name"].lower(), text) >= 90:
            out.append(("set", p["set_name"]))
    for amount in (truth.get("budget"), truth.get("asked")):
        if amount and f"{amount:g}" in text.split():
            out.append(("amount", f"{amount:g}"))
    return list(dict.fromkeys(out))

In [ ]:
#| export eval_voice
def entity_recall(query: dict, transcript: str) -> float | None:
    want = entities(query)
    if not want:
        return None
    text = " ".join(normalize(transcript))
    found = 0
    for kind, value in want:
        if kind == "card":
            found += name_matches(value, text, 85)
        elif kind == "set":
            found += fuzz.partial_ratio(value.lower(), text) >= 85
        else:
            found += value in text.split()
    return round(found / len(want), 3)

In [ ]:
#| export eval_voice
def spoken_audio(text: str, voice_name: str, config: dict[str, Any], synth=None) -> bytes:
    """Edge-TTS audio of a request in one voice, cached in .cache/eval_audio."""
    from src.voice import Voice

    key = hashlib.sha256(f"{voice_name}|{text}".encode()).hexdigest()[:16]
    path = repo_path(".cache/eval_audio") / f"{key}.mp3"
    if path.exists():
        return path.read_bytes()
    cfg = copy.deepcopy(config)
    cfg["voice"]["tts_voice"] = voice_name
    audio = (synth or Voice(cfg).edge_tts)(text)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(audio)
    return audio

In [ ]:
#| export eval_voice
def transcribers(config: dict[str, Any], bench: Benchmark, groq=None, gemini=None) -> dict:
    """The voice chat's Transcriber with the vocabulary prompt, and without it."""
    from src.voice import Transcriber

    names = sorted({x.card_name for x in bench.data.listings} | {x.set_name for x in bench.data.listings if x.set_name})  # fmt: skip
    bare = copy.deepcopy(config)
    bare["voice"]["stt_vocabulary"] = []
    return {"vocabulary": Transcriber(config, groq, gemini, names=names),
            "no_vocabulary": Transcriber(bare, groq, gemini)}  # fmt: skip

In [ ]:
#| export eval_voice
def run_stt(
    config: dict[str, Any],
    split: str = "dev",
    voices: list[str] | None = None,
    limit: int | None = None,
    agents_model: str | None = None,
    bench: Benchmark | None = None,
    stt: dict | None = None,
    synth=None,
    out_dir: Path | None = None,
    log=print,
) -> Path:
    """Speech to text (and, with `agents_model`, voice to voice) on the benchmark requests."""
    from src.evaluation import MultiAgent, eval_llm, fresh_db, score, variant_config, variants
    from src.memory import BuyerMemory
    from src.voice import Recording

    bench = bench or load_benchmark(config)
    voices = voices or list(config["evaluation"]["voices"])
    stt = stt or transcribers(config, bench)
    queries = [q for q in bench.queries if split == "all" or q["split"] == split][: limit or None]
    folder = out_dir or repo_path(config["evaluation"]["results_dir"]) / (
        datetime.now().strftime("%Y%m%d_%H%M%S") + "_voice"
    )
    folder.mkdir(parents=True, exist_ok=True)
    agents = None
    if agents_model:
        v = variants(config)["full"]
        cfg = variant_config(config, v)
        agents = MultiAgent(cfg, v, bench.data, eval_llm(cfg, agents_model, 0))
    typed: dict[str, dict] = {}
    rows = []
    for q in queries:
        if agents is not None:
            out = agents.ask(agents.session(BuyerMemory(fresh_db()), f"typed-{q['id']}"),
                             q["text"], q.get("clarify_answer"))  # fmt: skip
            typed[q["id"]] = score(q, out, bench.data, config)
        for voice_name in voices:
            audio = spoken_audio(q["text"], voice_name, config, synth)
            for arm, transcriber in stt.items():
                t0 = time.perf_counter()
                try:
                    text, service = transcriber.transcribe(
                        Recording(audio, "request.mp3", time.perf_counter()))  # fmt: skip
                except Exception as e:  # noqa: BLE001 - recorded, the run goes on
                    text, service = "", f"error: {e}"[:120]
                row = {"query_id": q["id"], "voice": voice_name, "arm": arm, "service": service,
                       "reference": q["text"], "transcript": text,
                       "stt_s": round(time.perf_counter() - t0, 3), "wer": wer(q["text"], text),
                       "entity_recall": entity_recall(q, text)}  # fmt: skip
                if agents is not None and arm == "vocabulary" and text:
                    out = agents.ask(agents.session(BuyerMemory(fresh_db()),
                                                    f"voice-{q['id']}-{voice_name}"),
                                     text, q.get("clarify_answer"))  # fmt: skip
                    spoken = score(q, out, bench.data, config)
                    row["voice_top1"] = spoken["top1_accurate"]
                    row["typed_top1"] = typed[q["id"]]["top1_accurate"]
                    row["same_pick"] = int(spoken["top_pick_id"] == typed[q["id"]]["top_pick_id"])
                rows.append(row)
                log(f"{q['id']} {voice_name:22} {arm:13} wer={row['wer']:.2f} "
                    f"entities={row['entity_recall']} {row['stt_s']:.2f}s")  # fmt: skip
    cols = list(dict.fromkeys(k for r in rows for k in r))
    with (folder / "voice.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, cols)
        w.writeheader()
        w.writerows(rows)
    (folder / "voice_summary.json").write_text(json.dumps(summarize_stt(rows), indent=1))
    return folder

In [ ]:
#| export eval_voice
def summarize_stt(rows: list[dict]) -> dict[str, Any]:
    def mean(xs):
        xs = [x for x in xs if x is not None and x != ""]
        return round(statistics.fmean(float(x) for x in xs), 4) if xs else None

    out: dict[str, Any] = {}
    for arm in sorted({r["arm"] for r in rows}):
        ar = [r for r in rows if r["arm"] == arm]
        out[arm] = {"transcripts": len(ar), "wer": mean(r["wer"] for r in ar),
                    "entity_recall": mean(r["entity_recall"] for r in ar),
                    "stt_s_median": round(statistics.median(r["stt_s"] for r in ar), 3),
                    "by_voice": {v: {"wer": mean(r["wer"] for r in ar if r["voice"] == v),
                                     "entity_recall": mean(r["entity_recall"] for r in ar
                                                           if r["voice"] == v)}
                                 for v in sorted({r["voice"] for r in ar})}}  # fmt: skip
        if any("voice_top1" in r for r in ar):
            out[arm]["voice_top1"] = mean(r.get("voice_top1") for r in ar)
            out[arm]["typed_top1"] = mean(r.get("typed_top1") for r in ar)
            out[arm]["same_pick"] = mean(r.get("same_pick") for r in ar)
    return out

In [ ]:
#| export eval_voice
def timing_summary(log_path: Path) -> dict[str, Any]:
    """H5 from the voice chat's timing log: does the filler come well before the answer?"""
    rows = [json.loads(x) for x in log_path.read_text().splitlines() if x.strip()]
    searched = [r for r in rows if r.get("to_filler_s") is not None]
    gaps = [{"gap": r["to_answer_s"] - r["to_filler_s"]} for r in searched if r.get("to_answer_s")]

    def med(col, rs):
        xs = [r[col] for r in rs if r.get(col) is not None]
        return round(statistics.median(xs), 2) if xs else None

    return {
        "turns": len(rows),
        "turns_with_filler": len(searched),
        "to_transcript_s": med("to_transcript_s", rows),
        "to_filler_s": med("to_filler_s", searched),
        "to_answer_s": med("to_answer_s", searched),
        "to_answer_s_all_turns": med("to_answer_s", rows),
        "filler_before_answer_s": med("gap", gaps),
        "graph_s": med("graph_s", rows),
        "tts_first_sentence_s": med("tts_s", rows),
        "models": sorted({r.get("model", "") for r in rows}),
    }

In [ ]:
#| export eval_voice
def tts_latency(texts: list[str], config: dict[str, Any], synth=None) -> dict[str, Any]:
    """Edge-TTS time for the first sentence of each reply (what the buyer waits for)."""
    from src.voice import Voice, sentences, speakable

    voice = Voice(config, synth=synth)
    times, failures = [], []
    for text in texts:
        first = sentences(speakable(text))[0]
        t0 = time.perf_counter()
        try:
            voice.mp3(first)
        except Exception as e:  # noqa: BLE001 - a failure is counted, not timed
            failures.append(f"{type(e).__name__}: {e}"[:120])
        else:
            times.append(time.perf_counter() - t0)
    return {"replies": len(texts), "made": len(times), "failures": failures[:5],
            "median_s": round(statistics.median(times), 3) if times else None,
            "p90_s": round(sorted(times)[int(0.9 * (len(times) - 1))], 3) if times else None}  # fmt: skip

In [ ]:
#| export eval_voice
def main(argv: list[str] | None = None) -> int:
    """`python -m src.eval_voice stt|timing|tts ...`"""
    ap = argparse.ArgumentParser(
        prog="python -m src.eval_voice", description=__doc__.split("\n")[0]
    )
    sub = ap.add_subparsers(dest="cmd", required=True)
    s = sub.add_parser("stt", help="speech to text (and voice to voice) on the benchmark")
    s.add_argument("--split", default="dev", choices=["dev", "test", "all"])
    s.add_argument("--voices", default="all", help="comma-separated Edge-TTS voices, or all")
    s.add_argument("--limit", type=int)
    s.add_argument("--agents", metavar="MODEL", help="also run the agents on each transcript")
    t = sub.add_parser("timing", help="H5 from the voice chat's timing log")
    t.add_argument("log", nargs="?", default=None)
    r = sub.add_parser("tts", help="Edge-TTS time to the first sentence of a run's replies")
    r.add_argument("folder")
    r.add_argument("--limit", type=int, default=20)
    args = ap.parse_args(argv)
    cfg = load_config()
    if args.cmd == "stt":
        voices = None if args.voices == "all" else args.voices.split(",")
        folder = run_stt(cfg, args.split, voices, args.limit, args.agents)
        print(f"\nresults: {folder}")
        print((folder / "voice_summary.json").read_text())
    elif args.cmd == "timing":
        print(
            json.dumps(timing_summary(repo_path(args.log or cfg["voice"]["timing_log"])), indent=1)
        )
    elif args.cmd == "tts":
        recs = [
            json.loads(x) for x in (Path(args.folder) / "replies.jsonl").read_text().splitlines()
        ]
        texts = [r["reply"] for r in recs if r.get("reply")][: args.limit]
        print(json.dumps(tts_latency(texts, cfg), indent=1))
    return 0

In [ ]:
#| export eval_voice
if __name__ == "__main__" and "ipykernel" not in sys.modules:  # not inside a notebook
    sys.exit(main())

# Try it

Offline: the benchmark, the scam check on every listing, a small run of three variants on the dev
queries with the rule-based stand-ins, the pipeline on the labeled samples, and the word error
rate. No keys, no network; the results go to a temporary folder.

In [ ]:
import tempfile

import pandas as pd

from src.config import load_config

cfg = load_config()
bench = load_benchmark(cfg)
pool = pd.DataFrame([x.model_dump() for x in bench.data.listings])
pd.crosstab(pool["label"], pool["split"], margins=True)

Three answer keys: a simple request, a collector's (condition first), and one where nothing fits and a trap is the bait:

In [ ]:
for qid in ("S01", "C01", "C12"):
    q = next(q for q in bench.queries if q["id"] == qid)
    print(
        qid,
        q["text"],
        "->",
        q["key"]["expect"],
        q["key"].get("acceptable"),
        "bait:",
        q["key"].get("bait_traps"),
    )

The scam check on every listing: dev traps use the knowledge base's own words, test traps don't.

In [ ]:
pd.DataFrame(screening(cfg, bench)).T.drop(columns="missed_traps")

A small run (the rule-based stand-ins, so the numbers only show that everything is wired):

In [ ]:
folder = run(
    cfg,
    ["full", "single_agent", "no_reflection"],
    split="dev",
    limit=8,
    label="notebook",
    model_override="offline",
    out_dir=Path(tempfile.mkdtemp()),
    bench=bench,
    log=lambda _m: None,
)
print(summarize(folder)["report"].read_text())

The pipeline on the 28 labeled sample listings (offline identification):

In [ ]:
items = json.loads(repo_path("data/samples/sample_raw_listings.json").read_text())
result = score_pipeline(
    cfg, items, load_labels(repo_path("eval/data/sample_raw_labels.csv")), make_llm(cfg)
)
{k: v for k, v in result.items() if k not in ("per_listing", "identify")}

Word error rate and entity recall on a misheard request:

In [ ]:
q = next(q for q in bench.queries if q["id"] == "S09")
heard = "Lugia 5 alt art from silver tempest under $120"
print(q["text"], "|", heard, "| WER", wer(q["text"], heard), "| entities", entity_recall(q, heard))

## Export

Run this after you change any `#| export` cell. It rewrites the generated module in `src/`,
so the other notebooks, the graph and the tests use your new version.

In [ ]:
from src.nbexport import export_all

changed = export_all()